# GraphMS-Net — ResEncM-250 TRUE HYBRID v3.5.1 — FINAL GUIDE-LEAN AUDITED EXECUTION


> **v3.5.1 audit hotfix:** Stage-7 patience is restored to its audited 20-epoch value; fusion patience is restored to the established 20-epoch value. The final package-name variable is corrected. No segmentation method, architecture, loss, optimizer family, Stage-11 rule, or workload search is added.
## Binding priorities — in order

1. **Highest legitimate metric.**
2. **Stay inside the supplied GraphMS-Net guide method space.**
3. **Among solutions satisfying 1–2, minimize unnecessary compute/recomputation.**
4. Preserve validated expensive ResEncM-250 / graph / GAT work whenever provenance matches.
5. No outer-fold label may affect a model applied to that outer fold before prediction is frozen.
6. Recovery/storage engineering is not scientific methodology.

## Scope truth-lock

This notebook completes the **guide-contained segmentation branch**. It does not falsely claim that
all 16 stages are re-executed here.

- Stages 4–11, 15 and segmentation Stage 16 are active here.
- Stage 5/6 ResEncM assets are reused only after SHA verification.
- Stages 2–3 remain the frozen ResEncM/nnU-Net preprocessing+augmentation lineage; the notebook does
  **not** claim that ResEncM-250 was retrained under a different literal BET/N4/ANTs chain.
- Stages 12–14 follow after segmentation freezes.

## One fixed segmentation path — no search framework

`FLAIR/T1/T2`
→ frozen **ResEncM-250**
→ four-scale CNN features
→ patch graph
→ **GAT/message passing**
→ **CNN+GNN concatenation**
→ **SE attention**
→ **self-attention**
→ **multi-scale fusion**
→ transposed-convolution decoder + skip
→ direct **1×1×1 lesion head**
→ **Dice+BCE**
→ **validated Stage-11 setting: p=0.5 + 26-connected components + filter5**
→ DSC / IoU / sensitivity / specificity / HD95
→ **5-fold mean ± sample SD**.

## Workload lock

The guide defines the method family but does not require the earlier 60×1000 fusion-training budget.
For the final execution:

- fusion epoch = **250 stochastic patch iterations**;
- maximum = **60 epochs**;
- validation = **every 5 epochs**;
- stop after **20 epochs without a new best inner Dice**;
- recovery checkpoint = **every completed epoch**;
- final refit uses the **same 250-iteration epoch definition and the same 60-epoch cosine horizon**.

Because `250` is not divisible by gradient accumulation `8`, the last partial accumulation group is
normalized by its **actual group length**. No partial optimizer step is underweighted.

## Stage-11 truth-lock

The guide lists thresholding, connected components, morphology, and small-lesion removal as the
Stage-11 method family. The authoritative master audit does **not** prescribe one mandatory numeric
threshold/component cutoff.

This notebook therefore uses the already validated project setting:

- threshold **0.5**;
- **26-connectivity**;
- remove connected components smaller than **5 voxels** (`filter5`).

`filter5` is a **validated implementation setting**, not a claimed guide-mandated number.
No new Stage-11 sweep is performed and no morphology parameters are invented.

## Explicitly absent

No fusion/loss/optimizer/scheduler tournament.
No Stage-11 parameter sweep.
No CATMIL / CutMix / DBL / CC-DiceCE.
No meta-gating / component verifier / probability-rescue machinery.
No label smoothing / stochastic depth / DropPath / AMSGrad / LR warmup.
No bootstrap/Wilcoxon/patient-macro promotion gate.

## Necessary validation hygiene only

For each outer fold:

- train Stage 8 on three folds;
- use one non-outer inner fold only for epoch selection/early stopping;
- refit the same fixed model on all four non-outer folds for the selected epoch count;
- predict the outer fold once.

This is validation hygiene, not an additional GraphMS-Net module.


In [ ]:
#@title 0. Bootstrap — T4 + mounted Drive + LOCAL compute staging
import os, sys, subprocess, importlib.util, json, io, time, math, random, hashlib, shutil, gc, pickle
import socket, errno
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, OrderedDict
from importlib.metadata import version as dist_version

def ensure_pkg(mod,spec):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",spec])

ensure_pkg("SimpleITK","SimpleITK>=2.5")
ensure_pkg("nnunetv2","nnunetv2==2.8.1")

import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
import SimpleITK as sitk

import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(),"Select a GPU runtime. T4 is sufficient."
DEVICE=torch.device("cuda")
GPU_NAME=torch.cuda.get_device_name(0)

from google.colab import drive
DRIVE_MOUNT=Path("/content/drive")
drive.mount(str(DRIVE_MOUNT),force_remount=False)

MYDRIVE=DRIVE_MOUNT/"MyDrive"
PROJECT_MOUNT=MYDRIVE/"MSLesSeg_MS"
assert PROJECT_MOUNT.is_dir(),f"Expected project root not found: {PROJECT_MOUNT}"

from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.inference.export_prediction import convert_predicted_logits_to_segmentation_with_correct_shape
from nnunetv2.inference.sliding_window_prediction import compute_gaussian
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from acvl_utils.cropping_and_padding.padding import pad_nd_image

torch.backends.cudnn.benchmark=True
os.environ["nnUNet_compile"]="false"

print("GPU:",GPU_NAME)
print("torch:",torch.__version__,"CUDA build:",torch.version.cuda)
print("nnU-Net:",dist_version("nnunetv2"))
print("Persistent storage: mounted Google Drive")
print("Training/inference workspace: /content local disk")


In [ ]:
#@title 1. Protocol — frozen Stage5/6 + v3.5.1 FINAL GUIDE-LEAN audited downstream

IDS={
    "project":"1X70rXCY4pv_vU2CvUznYquPhAAnSiYTa",
    "nnunet_v2":"1nD_XZHkZBbBHpSbRP9LwTz3-vpyBGvNn",
    "stage6":"1jV5ZX5h_yC6Fx1mK-h2J7s72ysZnY22l",
    "stage6_manifests":"1ZLeggsUnPjuhBcWHU5qOdD1aIWSn8dPo",
    "results_dataset001":"1gzTazBFnVbbxH8BaKk_rfVE2g7uBvM4b",
    "resencm250":"1ZI20FlZ8iWB89sdAba3KSqZp4TSTsiSv",
    "raw_dataset001":"1-rQmLBZ3q1LORfd-UvSx7m7sOObQMUQk",
    "labelsTr":"1mfgst8VBXCTB5ObwpbpNd4hLK6dYvcsw",
}

OUTER_FOLDS_TO_RUN=[1,2,3,4]

# Frozen Stage5/6 definition — must remain compatible with already-built artifacts.
CELL_SIZE=4
BRAIN_CELL_MIN_FRACTION=0.001
MAX_NODES_OPERATIONAL=90000
SPATIAL_KNN=8
FEATURE_KNN=4
SEMANTIC_DEEP_GROUPS=16
ENCODER_STAGE_INDICES=(-4,-3,-2,-1)

# Stage7 graph model.
GAT_HIDDEN=128
GAT_HEADS=4
GAT_MAX_TUNE_EPOCHS=60
GAT_EVAL_EVERY=5
GAT_ACCUM_CASES=8

# Stage8/9.
FPN_CHANNELS=32
PATCH_VOXELS=64
PATCH_CELLS=PATCH_VOXELS//CELL_SIZE
BATCH_SIZE=2
GRAD_ACCUM_STEPS=8
FUSION_MAX_TUNE_EPOCHS=60
FUSION_ITERS_PER_EPOCH=250
FUSION_EVAL_EVERY=5

BASE_SEED=20260911

# ------------------------------------------------------------------
# Exact frozen upstream protocol identity from v3.1 Stage5/6.
# The legacy values below are retained ONLY to reproduce the existing
# Stage5/6 artifact SHA and do not define the new downstream method.
# ------------------------------------------------------------------
LEGACY_PROTOCOL={
    "code_revision":"graphms-resencm250-root-correct-v3.1-audited",
    "cnn":"ResEncM-250 fold-specific model for complete outer-fold pipeline",
    "modalities":["FLAIR","T1","T2"],
    "encoder_stage_indices":ENCODER_STAGE_INDICES,
    "graph":{
        "cell_size":CELL_SIZE,
        "brain_cell_min_fraction":BRAIN_CELL_MIN_FRACTION,
        "spatial_knn":SPATIAL_KNN,
        "feature_knn":FEATURE_KNN,
        "semantic_deep_groups":SEMANTIC_DEEP_GROUPS,
        "max_nodes_operational_fail_closed":MAX_NODES_OPERATIONAL,
    },
    "gat":{
        "family":"GAT","hidden":128,"heads":4,"dropout":0.30,
        "identity_candidate":False,
    },
    "stage8":"CNN scale grids + real GAT grids -> concat + SE + self-attention + spatial FPN",
    "stage9":"FPN -> transposed-conv decoder -> CNN-logit skip -> direct 1x1x1 final lesion logit",
    "loss":{
        "mode":"dice_bce","dice_weight":0.5,"bce_weight":0.5,"label_smoothing":0.10,
    },
    "optimizer":{
        "type":"AdamW","lr":1e-4,"betas":(0.9,0.999),"eps":1e-8,
        "amsgrad":True,"weight_decay":1e-4,"warmup_epochs":5,"min_lr":1e-6,
    },
    "patch":{
        "voxels":64,"batch_size":2,"grad_accum":8,"iterations_per_epoch":1000,
    },
    "storage":{
        "baseline_logits":"float32",
        "edge_attributes":"float32",
        "deep_encoder_features":"float16_from_FP16_training/inference_path",
    },
    "stage11":{
        "threshold":0.5,"remove_components_lt_voxels":10,
        "morphology":"not parameterized because supplied guide gives no morphology parameters",
    },
    "claim_scope":"development 5-fold CV",
}
LEGACY_FEATURE_PROTOCOL_SHA=hashlib.sha256(
    json.dumps(LEGACY_PROTOCOL,sort_keys=True,separators=(",",":")).encode()
).hexdigest()
LEGACY_FEATURE_ROOT_NAME=f"graphms_resencm250_true_hybrid_v3_1_{LEGACY_FEATURE_PROTOCOL_SHA[:10]}"

assert LEGACY_FEATURE_PROTOCOL_SHA=="1b2e14c1fe9e8f7e462241412d19a0402a3a524df6736a2237d339205017674f"
assert LEGACY_FEATURE_ROOT_NAME=="graphms_resencm250_true_hybrid_v3_1_1b2e14c1fe"


# ------------------------------------------------------------------
# v3.5.1 FINAL GUIDE-LEAN audited downstream.
# Stage7 science is unchanged from v3.4.1; Stage8 workload and Stage11 validated
# implementation setting change, so the downstream protocol gets a NEW namespace.
# ------------------------------------------------------------------
GUIDE_DROPOUT=0.30
GUIDE_WEIGHT_DECAY=1e-4
GUIDE_LR=1e-4
GUIDE_BETAS=(0.9,0.999)
GUIDE_EPS=1e-8
GUIDE_MIN_LR=1e-6
GAT_EARLY_STOP_PATIENCE_EPOCHS=20
FUSION_EARLY_STOP_PATIENCE_EPOCHS=20

GUIDE_THRESHOLD=0.5
GUIDE_MIN_COMPONENT_VOXELS=5
GUIDE_CONNECTIVITY=26

# Stage7 scientific implementation is intentionally unchanged so exact compatible
# completed Stage7 checkpoints can be reused after provenance verification.
STAGE7_IMPL_ID="v3.4.1-gat-fp32-dice-bce-adamw-cosine-baseline-preserving-selection"
FUSION_IMPL_ID="v3.5-concat-se-self-multiscale-dice-bce-adamw-cosine-250iter"

DOWNSTREAM_PROTOCOL={
    "code_revision":"graphms-resencm250-true-hybrid-v3.5.1-final-guide-lean-audited",
    "frozen_feature_protocol_sha":LEGACY_FEATURE_PROTOCOL_SHA,
    "priorities":[
        "highest legitimate metric",
        "guide method boundary",
        "minimum unnecessary compute/recompute",
    ],
    "pipeline":{
        "stage7":"GAT/message passing",
        "stage7_training":{"max_epochs":GAT_MAX_TUNE_EPOCHS,"validation_every_epochs":GAT_EVAL_EVERY,"early_stop_patience_epochs":GAT_EARLY_STOP_PATIENCE_EPOCHS},
        "stage8":"CNN+GNN concat -> SE -> self-attention -> multi-scale fusion",
        "stage9":"transposed-convolution decoder + CNN-logit skip + final 1x1x1 lesion head",
        "stage10":"Dice+BCE",
        "stage11":{
            "threshold":GUIDE_THRESHOLD,
            "connected_components":"3D 26-connectivity",
            "remove_components_lt_voxels":GUIDE_MIN_COMPONENT_VOXELS,
            "setting_source":"previously validated project filter5; not claimed guide-mandated",
            "morphology":"guide-listed family; no unspecified morphology parameters invented",
            "parameter_search":False,
        },
        "stage15":{
            "optimizer":"AdamW",
            "lr":GUIDE_LR,
            "betas":GUIDE_BETAS,
            "eps":GUIDE_EPS,
            "weight_decay":GUIDE_WEIGHT_DECAY,
            "schedule":"cosine",
            "cosine_horizon_epochs":FUSION_MAX_TUNE_EPOCHS,
            "min_lr":GUIDE_MIN_LR,
            "dropout":GUIDE_DROPOUT,
            "patch_voxels":PATCH_VOXELS,
            "batch_size":BATCH_SIZE,
            "gradient_accumulation":GRAD_ACCUM_STEPS,
            "iterations_per_epoch":FUSION_ITERS_PER_EPOCH,
            "partial_accumulation_normalization":"actual group length",
            "max_epochs":FUSION_MAX_TUNE_EPOCHS,
            "validation_every_epochs":FUSION_EVAL_EVERY,
            "early_stop_patience_epochs":FUSION_EARLY_STOP_PATIENCE_EPOCHS,
            "checkpoint_every_completed_epoch":True,
        },
        "stage16":["DSC","IoU","Sensitivity","Specificity","HD95_mm"],
    },
    "no_search":{
        "fusion_tournament":False,
        "loss_tournament":False,
        "optimizer_tournament":False,
        "scheduler_tournament":False,
        "stage11_grid_search":False,
    },
    "non_guide_extras":{
        "label_smoothing":False,
        "stochastic_depth":False,
        "amsgrad":False,
        "lr_warmup":False,
        "CATMIL":False,
        "CutMix":False,
        "DBL":False,
        "CC_DiceCE":False,
        "meta_gating":False,
        "component_verifier":False,
    },
    "nesting":{
        "stage7_tune":"train3 -> inner",
        "stage7_refit":"four non-outer folds",
        "stage8_tune":"train3 -> inner using tune GAT",
        "stage8_refit":"four non-outer folds using refit GAT",
        "outer_gt_used_before_prediction":False,
    },
    "gat_uncovered_voxels":"ResEncM-250 baseline",
    "gat_epoch_selection_uncovered_voxels":"ResEncM-250 baseline",
    "scope_truth":{
        "stages_4_11_15_16_segmentation":"active",
        "stages_2_3":"frozen ResEncM/nnU-Net lineage; no false literal BET/N4/ANTs claim",
        "stages_12_14":"after segmentation freeze",
    },
    "claim_scope":"development 5-fold CV",
}
DOWNSTREAM_PROTOCOL_SHA=hashlib.sha256(
    json.dumps(DOWNSTREAM_PROTOCOL,sort_keys=True,separators=(",",":")).encode()
).hexdigest()
DOWNSTREAM_REMOTE_EXPERIMENT_NAME=f"graphms_resencm250_true_hybrid_v3_5_1_{DOWNSTREAM_PROTOCOL_SHA[:10]}"

PROTOCOL_SHA=DOWNSTREAM_PROTOCOL_SHA

LOCAL=Path("/content/graphms_resencm250_true_hybrid_v3_5_1")
MODEL_LOCAL=LOCAL/"resenc250_model"
RAW_LOCAL=LOCAL/"raw_tmp"
PREP_LOCAL=LOCAL/"preprocessed"
FOLD_LOCAL=LOCAL/"outer_folds"
REPORT_LOCAL=LOCAL/"reports"
for p in [MODEL_LOCAL,RAW_LOCAL,PREP_LOCAL,FOLD_LOCAL,REPORT_LOCAL]:
    p.mkdir(parents=True,exist_ok=True)

print("Frozen Stage5/6 SHA:",LEGACY_FEATURE_PROTOCOL_SHA)
print("Downstream v3.5.1 SHA:",DOWNSTREAM_PROTOCOL_SHA)
print("Downstream namespace:",DOWNSTREAM_REMOTE_EXPERIMENT_NAME)

def seed_all(seed):
    seed=int(seed)
    random.seed(seed);np.random.seed(seed)
    torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)

def stable_seed(*parts):
    b="|".join(map(str,parts)).encode()
    return BASE_SEED + int(hashlib.sha256(b).hexdigest()[:8],16)%1_000_000

def capture_rng():
    return {
        "python":random.getstate(),
        "numpy":np.random.get_state(),
        "torch_cpu":torch.get_rng_state(),
        "torch_cuda":torch.cuda.get_rng_state_all(),
    }

def _cpu_byte_tensor(x):
    if torch.is_tensor(x):
        return x.detach().to(device="cpu",dtype=torch.uint8).contiguous()
    return torch.as_tensor(x,dtype=torch.uint8,device="cpu").contiguous()

def restore_rng(state):
    if not state:return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(_cpu_byte_tensor(state["torch_cpu"]))
    if torch.cuda.is_available():
        states=[_cpu_byte_tensor(x) for x in state.get("torch_cuda",[])]
        ndev=torch.cuda.device_count()
        if len(states)==ndev:
            torch.cuda.set_rng_state_all(states)
        else:
            for dev_idx,x in enumerate(states[:ndev]):
                torch.cuda.set_rng_state(x,device=dev_idx)

print("Frozen Stage5/6 SHA:",LEGACY_FEATURE_PROTOCOL_SHA)
print("Downstream v3.5.1 SHA:",DOWNSTREAM_PROTOCOL_SHA)
print("Downstream namespace:",DOWNSTREAM_REMOTE_EXPERIMENT_NAME)


In [ ]:
#@title 2. Mounted-Drive filesystem adapter — local-first, atomic sync, remount retry

FOLDER_MIME="application/vnd.google-apps.folder"
OPERATIONAL_BUILD="v3.2.2-mount-local-staging"
MOUNT_ATTEMPTS=6

# Original Drive IDs are retained in the scientific/provenance cells.
# This adapter maps them onto their mounted filesystem locations.
PATH_ID_MAP={}

def _is_mount_alive():
    try:
        return PROJECT_MOUNT.is_dir() and (PROJECT_MOUNT/"nnunet_v2").is_dir()
    except OSError:
        return False

def _remount():
    print("Drive mount unavailable -> remounting...")
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(2)
    drive.mount(str(DRIVE_MOUNT),force_remount=True)
    if not _is_mount_alive():
        raise RuntimeError("Google Drive remount did not restore project path")
    print("Drive remount PASS")

def _mount_call(fn,label,attempts=MOUNT_ATTEMPTS):
    last=None
    for k in range(1,attempts+1):
        try:
            if not _is_mount_alive():
                _remount()
            return fn()
        except (OSError,IOError,TimeoutError) as e:
            last=e
            if k==attempts:
                raise
            wait=min(20,2**min(k,4))
            print(f"{label}: mount I/O retry {k}/{attempts} after {wait}s | {type(e).__name__}: {e}")
            time.sleep(wait)
            _remount()
    raise RuntimeError(label) from last

def _dir_children_names(p):
    return {x.name for x in p.iterdir()} if p.is_dir() else set()

def _discover_dataset_dirs():
    """
    Resolve raw/results Dataset001_MSLesSeg without Google Drive API.
    First use conventional paths; if necessary, scan only the project tree.
    """
    nnroot=PROJECT_MOUNT/"nnunet_v2"
    raw_candidates=[
        nnroot/"nnUNet_raw"/"Dataset001_MSLesSeg",
        PROJECT_MOUNT/"nnUNet_raw"/"Dataset001_MSLesSeg",
    ]
    result_candidates=[
        nnroot/"nnUNet_results"/"Dataset001_MSLesSeg",
        PROJECT_MOUNT/"nnUNet_results"/"Dataset001_MSLesSeg",
    ]

    raw=next((p for p in raw_candidates if p.is_dir() and (p/"imagesTr").is_dir() and (p/"labelsTr").is_dir()),None)
    results=next((p for p in result_candidates if p.is_dir()),None)

    if raw is None or results is None:
        found=[]
        for base,dirs,files in os.walk(PROJECT_MOUNT):
            bp=Path(base)
            # keep fallback scan bounded
            try:
                rel=bp.relative_to(PROJECT_MOUNT)
                if len(rel.parts)>6:
                    dirs[:]=[]
                    continue
            except Exception:
                pass
            if bp.name=="Dataset001_MSLesSeg":
                found.append(bp)

        if raw is None:
            raw=next((p for p in found if (p/"imagesTr").is_dir() and (p/"labelsTr").is_dir()),None)

        model_name="nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres"
        if results is None:
            results=next((p for p in found if (p/model_name).is_dir()),None)

    if raw is None:
        raise FileNotFoundError("Could not locate mounted Dataset001_MSLesSeg containing imagesTr + labelsTr")
    if results is None:
        raise FileNotFoundError("Could not locate mounted nnUNet_results/Dataset001_MSLesSeg")

    return raw,results

RAW_DATASET_MOUNT,RESULTS_DATASET_MOUNT=_mount_call(
    _discover_dataset_dirs,"resolve Dataset001_MSLesSeg"
)

NNUNET_MOUNT=PROJECT_MOUNT/"nnunet_v2"
STAGE6_MOUNT=NNUNET_MOUNT/"graphms_stage6_oof_v2"
STAGE6_MANIFESTS_MOUNT=STAGE6_MOUNT/"manifests"
RESENCM250_MOUNT=RESULTS_DATASET_MOUNT/"nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres"
LABELSTR_MOUNT=RAW_DATASET_MOUNT/"labelsTr"

for p in [
    NNUNET_MOUNT,STAGE6_MOUNT,STAGE6_MANIFESTS_MOUNT,
    RAW_DATASET_MOUNT,RESULTS_DATASET_MOUNT,RESENCM250_MOUNT,LABELSTR_MOUNT
]:
    assert p.is_dir(),p

PATH_ID_MAP.update({
    IDS["project"]:PROJECT_MOUNT,
    IDS["nnunet_v2"]:NNUNET_MOUNT,
    IDS["stage6"]:STAGE6_MOUNT,
    IDS["stage6_manifests"]:STAGE6_MANIFESTS_MOUNT,
    IDS["results_dataset001"]:RESULTS_DATASET_MOUNT,
    IDS["resencm250"]:RESENCM250_MOUNT,
    IDS["raw_dataset001"]:RAW_DATASET_MOUNT,
    IDS["labelsTr"]:LABELSTR_MOUNT,
})

def _as_path(ref):
    if isinstance(ref,Path):
        return ref
    s=str(ref)
    if s in PATH_ID_MAP:
        return Path(PATH_ID_MAP[s])
    # every filesystem "id" returned by this adapter is an absolute path string
    p=Path(s)
    if p.is_absolute():
        return p
    raise KeyError(f"Unknown mounted Drive reference: {ref}")

def _meta(p):
    p=Path(p)
    return {
        "id":str(p),
        "name":p.name,
        "mimeType":FOLDER_MIME if p.is_dir() else "application/octet-stream",
        "size":str(p.stat().st_size) if p.is_file() else None,
        "modifiedTime":str(p.stat().st_mtime),
        "parents":[str(p.parent)],
    }

def file_meta(fid):
    p=_as_path(fid)
    return _mount_call(lambda:_meta(p),f"metadata {p.name}")

def verify_id(fid,name=None,mime=None):
    x=file_meta(fid)
    if name is not None and x["name"]!=name:
        raise RuntimeError((fid,"expected name",name,"got",x["name"]))
    if mime is not None and x["mimeType"]!=mime:
        raise RuntimeError((fid,"expected mime",mime,"got",x["mimeType"]))
    return x

def children(pid):
    p=_as_path(pid)
    def op():
        return [_meta(x) for x in p.iterdir()]
    return _mount_call(op,f"list {p.name}")

def exact(pid,name,mime=None,required=True):
    p=_as_path(pid)/name
    def op():
        if not p.exists():
            return None
        if mime==FOLDER_MIME and not p.is_dir():
            return None
        return _meta(p)
    x=_mount_call(op,f"resolve {name}")
    if x is None and required:
        raise FileNotFoundError((str(_as_path(pid)),name))
    return x

def folder(pid,name,required=True):
    x=exact(pid,name,FOLDER_MIME,required)
    return None if x is None else x["id"]

def ensure_folder(pid,name):
    p=_as_path(pid)/name
    def op():
        p.mkdir(parents=True,exist_ok=True)
        return str(p)
    return _mount_call(op,f"mkdir {name}")

def _copy_mount_to_local(src,dest,force=False):
    src=Path(src);dest=Path(dest)
    dest.parent.mkdir(parents=True,exist_ok=True)
    if force:
        dest.unlink(missing_ok=True)
    if dest.exists() and dest.stat().st_size>0:
        return dest

    tmp=Path(str(dest)+".part")
    tmp.unlink(missing_ok=True)

    def op():
        if not src.is_file():
            raise FileNotFoundError(src)
        with src.open("rb") as fi,tmp.open("wb") as fo:
            shutil.copyfileobj(fi,fo,length=8*1024*1024)
            fo.flush();os.fsync(fo.fileno())
        if tmp.stat().st_size!=src.stat().st_size:
            raise IOError(f"copy size mismatch {tmp.stat().st_size}!={src.stat().st_size}")
        os.replace(tmp,dest)
        return dest

    try:
        return _mount_call(op,f"stage {src.name}")
    finally:
        tmp.unlink(missing_ok=True)

def download(fid,dest,force=False,attempts=None):
    # historical function name preserved; operation is mounted Drive -> /content
    return _copy_mount_to_local(_as_path(fid),Path(dest),force=force)

def sha256_file(path,block=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(block)
            if not b:break
            h.update(b)
    return h.hexdigest()

def remote_file(pid,name):
    return exact(pid,name,required=False)

def _atomic_sync_to_mount(local,dest):
    local=Path(local);dest=Path(dest)
    dest.parent.mkdir(parents=True,exist_ok=True)
    # same-directory temporary file -> rename is atomic from the reader's perspective
    tmp=dest.with_name(dest.name+f".tmp.{os.getpid()}")
    tmp.unlink(missing_ok=True)

    def op():
        with local.open("rb") as fi,tmp.open("wb") as fo:
            shutil.copyfileobj(fi,fo,length=8*1024*1024)
            fo.flush();os.fsync(fo.fileno())
        if tmp.stat().st_size!=local.stat().st_size:
            raise IOError(f"sync size mismatch {tmp.stat().st_size}!={local.stat().st_size}")
        os.replace(tmp,dest)
        return _meta(dest)

    try:
        return _mount_call(op,f"sync {dest.name}")
    finally:
        try: tmp.unlink(missing_ok=True)
        except Exception: pass

def upload(local,pid,name=None,mime="application/octet-stream"):
    # historical function name preserved; operation is /content -> mounted Drive
    local=Path(local)
    dest=_as_path(pid)/(name or local.name)
    return _atomic_sync_to_mount(local,dest)

def write_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=Path(str(path)+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str))
    os.replace(tmp,path)
    return path

def remote_json(pid,name,local_dir=None):
    x=remote_file(pid,name)
    if not x:return None
    target_dir=Path(local_dir) if local_dir is not None else Path("/content/graphms_remote_json_cache")
    target_dir.mkdir(parents=True,exist_ok=True)
    p=target_dir/name
    p.unlink(missing_ok=True)
    download(x["id"],p)
    return json.loads(p.read_text())

def warm_folder_name_cache(pid,force=False):
    # No API index exists in v3.2.2; direct mounted child lookup is O(1) by pathname.
    return len(children(pid))

print("Mounted filesystem adapter READY")
print("Raw dataset:",RAW_DATASET_MOUNT)
print("Results dataset:",RESULTS_DATASET_MOUNT)
print("ResEncM-250:",RESENCM250_MOUNT)
print("Scientific protocol/namespace: UNCHANGED")


In [ ]:
#@title 3. Resolve ONLY the split/case registry from historical handoff; reject its CNN lineage

for fid,name in [
    (IDS["nnunet_v2"],"nnunet_v2"),
    (IDS["stage6"],"graphms_stage6_oof_v2"),
    (IDS["stage6_manifests"],"manifests"),
    (IDS["resencm250"],"nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres"),
    (IDS["raw_dataset001"],"Dataset001_MSLesSeg"),
    (IDS["labelsTr"],"labelsTr"),
]:
    verify_id(fid,name,FOLDER_MIME)

handoff_meta=exact(IDS["stage6_manifests"],"GRAPH_STAGE_HANDOFF_v2.json")
handoff_local=LOCAL/"split_source_GRAPH_STAGE_HANDOFF_v2.json"
download(handoff_meta["id"],handoff_local,force=True)
HANDOFF=json.loads(handoff_local.read_text())

# The handoff is used ONLY for immutable case/patient/fold assignment.
# Its CNN probabilities/features/checkpoints are explicitly forbidden downstream in v3.
print("Historical handoff CNN lineage:",HANDOFF.get("cnn_backbone"))
assert "CATMIL" in str(HANDOFF.get("cnn_backbone","")), "Unexpected handoff provenance."

CASES=pd.DataFrame([
    {"case":x["case"],"patient":x["patient"],"fold":int(x["fold"])}
    for x in HANDOFF["cases"]
]).sort_values(["fold","case"]).reset_index(drop=True)

assert len(CASES)==93 and CASES.case.nunique()==93 and set(CASES.fold)==set(range(5))
print("Reused information: case IDs, patient IDs, five-fold assignment ONLY")
display(CASES.groupby("fold").size().rename("cases").reset_index())

# Frozen upstream v3.1 artifact store.
legacy_root_meta=exact(IDS["nnunet_v2"],LEGACY_FEATURE_ROOT_NAME,FOLDER_MIME,required=True)
LEGACY_FEATURE_ROOT=legacy_root_meta["id"]
REMOTE_PREP=ensure_folder(LEGACY_FEATURE_ROOT,"preprocessed_resenc250_plans")
FEATURE_REMOTE_OUTERS=ensure_folder(LEGACY_FEATURE_ROOT,"outer_folds")

# NEW v3.2 downstream namespace. No v3.1 GAT/fusion artifact can enter it.
REMOTE_ROOT=ensure_folder(IDS["nnunet_v2"],DOWNSTREAM_REMOTE_EXPERIMENT_NAME)
REMOTE_OUTERS=ensure_folder(REMOTE_ROOT,"outer_folds")
REMOTE_REPORT=ensure_folder(REMOTE_ROOT,"reports")

lock={
    "frozen_feature_protocol_sha":LEGACY_FEATURE_PROTOCOL_SHA,
    "frozen_feature_root":LEGACY_FEATURE_ROOT_NAME,
    "downstream_protocol_sha":DOWNSTREAM_PROTOCOL_SHA,
    "reuse_scope":"preprocessing + ResEncM-250 Stage5/6 only",
    "reuse_v31_gat_or_fusion":False,
}
proto_path=REPORT_LOCAL/"V3_2_PROTOCOL_LOCK.json"
write_json(proto_path,lock)
upload(proto_path,REMOTE_REPORT,"V3_2_PROTOCOL_LOCK.json","application/json")

print("Mounted legacy feature root:",_as_path(LEGACY_FEATURE_ROOT))
print("Mounted downstream root:",_as_path(REMOTE_ROOT))


In [ ]:
#@title 4. Resolve raw modalities + all five ResEncM-250 checkpoints + modality contract

IMAGES_ID=folder(IDS["raw_dataset001"],"imagesTr")
image_map={x["name"]:x for x in children(IMAGES_ID) if x["name"].endswith(".nii.gz")}
label_map={x["name"]:x for x in children(IDS["labelsTr"]) if x["name"].endswith(".nii.gz")}

MODEL_META={}
plans_meta=exact(IDS["resencm250"],"plans.json")
dataset_meta=exact(IDS["resencm250"],"dataset.json")
download(plans_meta["id"],MODEL_LOCAL/"plans.json")
download(dataset_meta["id"],MODEL_LOCAL/"dataset.json")

dataset_contract=json.loads((MODEL_LOCAL/"dataset.json").read_text())
channels=dataset_contract.get("channel_names",{})
channels_ordered=[str(channels[str(i)]) for i in range(len(channels))]
norm=[x.strip().upper().replace("-","").replace("_","") for x in channels_ordered]
expected=["FLAIR","T1","T2"]
assert norm==expected, (
    "Dataset channel order is not the locked FLAIR/T1/T2 order. "
    f"Found {channels_ordered}; expected {expected}. STOP before inference."
)
print("Dataset channel contract PASS:",channels_ordered)

for fold_idx in range(5):
    ff=folder(IDS["resencm250"],f"fold_{fold_idx}")
    ck=exact(ff,"checkpoint_final.pth")
    d=MODEL_LOCAL/f"fold_{fold_idx}";d.mkdir(exist_ok=True)
    p=d/"checkpoint_final.pth"
    download(ck["id"],p)
    MODEL_META[fold_idx]={
        "folder_id":ff,
        "checkpoint_id":ck["id"],
        "checkpoint_sha256":sha256_file(p),
    }

REG=[]
for r in CASES.itertuples():
    c=r.case
    names=[f"{c}_0000.nii.gz",f"{c}_0001.nii.gz",f"{c}_0002.nii.gz"]
    if not all(n in image_map for n in names):
        raise FileNotFoundError((c,names))
    gt=f"{c}.nii.gz"
    if gt not in label_map:raise FileNotFoundError(gt)
    REG.append({
        "case":c,"patient":r.patient,"fold":int(r.fold),
        "flair_id":image_map[names[0]]["id"],
        "t1_id":image_map[names[1]]["id"],
        "t2_id":image_map[names[2]]["id"],
        "gt_id":label_map[gt]["id"],
    })

REG=pd.DataFrame(REG).sort_values(["fold","case"]).reset_index(drop=True)
RMAP={r.case:r for r in REG.itertuples()}

print("ResEncM-250 checkpoint SHA256:")
display(pd.DataFrame([
    {"fold":f,"checkpoint_sha256":MODEL_META[f]["checkpoint_sha256"]}
    for f in range(5)
]))
print("Raw registry 93/93: PASS")


In [ ]:
#@title 5. Load a verified ResEncM-250 fold predictor

def load_resenc_predictor(fold_idx):
    p=nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=True,
        perform_everything_on_device=True,
        device=DEVICE,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=False,
    )
    p.initialize_from_trained_model_folder(
        str(MODEL_LOCAL),
        use_folds=(int(fold_idx),),
        checkpoint_name="checkpoint_final.pth",
    )
    assert "ResEnc" in p.network.__class__.__name__ or "UNet" in p.network.__class__.__name__
    ps=tuple(map(int,p.configuration_manager.patch_size))
    print("Loaded ResEncM-250 fold",fold_idx,"patch_size",ps)
    return p

_smoke=load_resenc_predictor(0)
print("Network class:",_smoke.network.__class__.__name__)
del _smoke
torch.cuda.empty_cache()


## Shared preprocessing cache

Preprocessing depends on the ResEncM-250 plans, not on fold weights. It is therefore performed once per
case and reused by all five outer-fold CNN branches.

The cache contains:

- preprocessed 3-channel FLAIR/T1/T2 tensor;
- preprocessed GT;
- nnU-Net properties required for exact inverse geometry.

No M150/CATMIL probability, feature, checkpoint, calibration, or mask is copied into this cache.


In [ ]:
#@title 6. Build/reuse 93-case ResEncM-250-plan preprocessing cache

PREP_DATA=PREP_LOCAL/"data"
PREP_PROPS=PREP_LOCAL/"props"
PREP_DATA.mkdir(exist_ok=True);PREP_PROPS.mkdir(exist_ok=True)

REMOTE_PREP_DATA=ensure_folder(REMOTE_PREP,"data")
REMOTE_PREP_PROPS=ensure_folder(REMOTE_PREP,"props")
REMOTE_PREP_COMMITS=ensure_folder(REMOTE_PREP,"commits")

def prep_npz(case):return PREP_DATA/f"{case}.npz"
def prep_pkl(case):return PREP_PROPS/f"{case}.pkl"
def prep_commit_name(case):return f"{case}.commit.json"

def stage_raw_case(case):
    r=RMAP[case]
    d=RAW_LOCAL/case;d.mkdir(parents=True,exist_ok=True)
    paths=[
        d/f"{case}_0000.nii.gz",
        d/f"{case}_0001.nii.gz",
        d/f"{case}_0002.nii.gz",
        d/f"{case}.nii.gz",
    ]
    for fid,p in zip([r.flair_id,r.t1_id,r.t2_id,r.gt_id],paths):
        download(fid,p)
    return paths

def try_reuse_preprocessed(case):
    cmeta=remote_file(REMOTE_PREP_COMMITS,prep_commit_name(case))
    nmeta=remote_file(REMOTE_PREP_DATA,f"{case}.npz")
    pmeta=remote_file(REMOTE_PREP_PROPS,f"{case}.pkl")
    if not (cmeta and nmeta and pmeta):return False

    cp=PREP_LOCAL/f"{case}.commit.json"
    if not cp.exists():
        download(cmeta["id"],cp)
    c=json.loads(cp.read_text())
    if c.get("protocol_sha256")!=PROTOCOL_SHA:return False

    n=prep_npz(case);p=prep_pkl(case)

    if not n.exists():download(nmeta["id"],n)
    if not p.exists():download(pmeta["id"],p)

    ok=(
        sha256_file(n)==c.get("npz_sha256")
        and sha256_file(p)==c.get("props_sha256")
    )
    if not ok:
        n.unlink(missing_ok=True);p.unlink(missing_ok=True)
        download(nmeta["id"],n,force=True);download(pmeta["id"],p,force=True)
        ok=(
            sha256_file(n)==c.get("npz_sha256")
            and sha256_file(p)==c.get("props_sha256")
        )
    return ok

prep_predictor=load_resenc_predictor(0)
pre=prep_predictor.configuration_manager.preprocessor_class(verbose=False)

for i,case in enumerate(REG.case,1):
    if try_reuse_preprocessed(case):
        if i%10==0 or i==len(REG):print("preprocess reuse",i,"/93")
        continue

    flair,t1,t2,gt=stage_raw_case(case)
    data,seg,props=pre.run_case(
        [str(flair),str(t1),str(t2)],str(gt),
        prep_predictor.plans_manager,
        prep_predictor.configuration_manager,
        prep_predictor.dataset_json,
    )
    data=np.asarray(data,np.float32)
    gtpre=np.asarray(seg[0]>0,np.uint8)

    # float16 is safe for MRI context storage; CNN inference receives float32 after reload.
    n=prep_npz(case);tmp=Path(str(n)+".tmp.npz")
    np.savez_compressed(tmp,data=data.astype(np.float16),gt=gtpre)
    os.replace(tmp,n)
    with prep_pkl(case).open("wb") as f:pickle.dump(props,f,pickle.HIGHEST_PROTOCOL)

    commit={
        "case":case,"protocol_sha256":PROTOCOL_SHA,
        "npz_sha256":sha256_file(n),"props_sha256":sha256_file(prep_pkl(case)),
        "shape":list(gtpre.shape),
        "created_utc":datetime.now(timezone.utc).isoformat(),
    }
    cp=PREP_LOCAL/f"{case}.commit.json";write_json(cp,commit)
    upload(n,REMOTE_PREP_DATA,mime="application/x-npz")
    upload(prep_pkl(case),REMOTE_PREP_PROPS,mime="application/octet-stream")
    upload(cp,REMOTE_PREP_COMMITS,prep_commit_name(case),"application/json")

    shutil.rmtree(RAW_LOCAL/case,ignore_errors=True)
    if i%10==0 or i==len(REG):print("preprocess build",i,"/93")

del prep_predictor,pre
torch.cuda.empty_cache();gc.collect()
print("SHARED PREPROCESSING CACHE COMPLETE 93/93")


In [ ]:
#@title 7. Graph / multi-scale CNN feature helpers

def pad_to_cell(arr,cell,fill=0):
    shape=arr.shape[-3:]
    target=tuple(int(math.ceil(s/cell)*cell) for s in shape)
    pads=[(0,0)]*(arr.ndim-3)+[(0,target[i]-shape[i]) for i in range(3)]
    return np.pad(arr,pads,mode="constant",constant_values=fill),shape,target

def block_view_3d(arr,cell):
    d,h,w=arr.shape[-3:];gd,gh,gw=d//cell,h//cell,w//cell
    prefix=arr.shape[:-3]
    x=arr.reshape(*prefix,gd,cell,gh,cell,gw,cell)
    axes=list(range(len(prefix)))+[len(prefix),len(prefix)+2,len(prefix)+4,
                                   len(prefix)+1,len(prefix)+3,len(prefix)+5]
    return x.transpose(axes)

def cell_statistics(data,pfg,cell):
    data_p,orig,target=pad_to_cell(data,cell,0)
    p_p,_,_=pad_to_cell(pfg,cell,0)
    brain=np.any(np.abs(data)>1e-6,axis=0)
    brain_p,_,_=pad_to_cell(brain.astype(np.uint8),cell,0)
    pb=block_view_3d(p_p,cell);bb=block_view_3d(brain_p,cell).astype(bool);db=block_view_3d(data_p,cell)
    pmax=pb.max(axis=(-3,-2,-1));pmean=pb.mean(axis=(-3,-2,-1));pstd=pb.std(axis=(-3,-2,-1))
    q=np.clip(pb,1e-6,1-1e-6);ent=-(q*np.log(q)+(1-q)*np.log(1-q))
    modality=[]
    for c in range(data.shape[0]):
        vals=db[c];cnt=bb.sum(axis=(-3,-2,-1));s=(vals*bb).sum(axis=(-3,-2,-1))
        mean=np.divide(s,cnt,out=np.zeros_like(s,dtype=np.float32),where=cnt>0)
        sq=((vals**2)*bb).sum(axis=(-3,-2,-1))
        var=np.divide(sq,cnt,out=np.zeros_like(sq,dtype=np.float32),where=cnt>0)-mean**2
        std=np.sqrt(np.maximum(var,0));vmax=np.where(bb,vals,-np.inf).max(axis=(-3,-2,-1));vmax[~np.isfinite(vmax)]=0
        modality.append((mean.astype(np.float32),std.astype(np.float32),vmax.astype(np.float32)))
    return {
        "orig_shape":orig,"target_shape":target,
        "pmax":pmax.astype(np.float32),"pmean":pmean.astype(np.float32),"pstd":pstd.astype(np.float32),
        "ent_mean":ent.mean(axis=(-3,-2,-1)).astype(np.float32),
        "ent_max":ent.max(axis=(-3,-2,-1)).astype(np.float32),
        "brain_frac":bb.mean(axis=(-3,-2,-1)).astype(np.float32),
        "modality":modality,
    }

def candidate_grid_coords(stats):
    coords=np.argwhere(stats["brain_frac"]>BRAIN_CELL_MIN_FRACTION).astype(np.int32)
    if len(coords)>MAX_NODES_OPERATIONAL:
        raise RuntimeError(
            f"{len(coords)} full-brain nodes exceed operational ceiling {MAX_NODES_OPERATIONAL}; "
            "nodes were NOT truncated."
        )
    assert len(coords)>0
    return coords

def node_bounds_centers(coords,shape):
    bounds=[];centers=[]
    for gz,gy,gx in coords:
        z0,y0,x0=int(gz*CELL_SIZE),int(gy*CELL_SIZE),int(gx*CELL_SIZE)
        z1,y1,x1=min(z0+CELL_SIZE,shape[0]),min(y0+CELL_SIZE,shape[1]),min(x0+CELL_SIZE,shape[2])
        bounds.append((z0,z1,y0,y1,x0,x1))
        centers.append(((z0+z1-1)/2,(y0+y1-1)/2,(x0+x1-1)/2))
    return np.asarray(bounds,np.int32),np.asarray(centers,np.float32)

def node_payload(logit_diff,gt,data,coords):
    K=CELL_SIZE**3
    logits=np.zeros((len(coords),K),np.float32)
    labels=np.zeros((len(coords),K),np.uint8)
    valid=np.zeros((len(coords),K),np.uint8)
    mri=np.zeros((len(coords),3,K),np.float16)
    bounds,centers=node_bounds_centers(coords,gt.shape)
    for i,(z0,z1,y0,y1,x0,x1) in enumerate(bounds):
        l=logit_diff[z0:z1,y0:y1,x0:x1].reshape(-1);g=gt[z0:z1,y0:y1,x0:x1].reshape(-1);n=len(l)
        logits[i,:n]=l;labels[i,:n]=g;valid[i,:n]=1
        for m in range(3):
            mri[i,m,:n]=data[m,z0:z1,y0:y1,x0:x1].reshape(-1).astype(np.float16)
    return logits,labels,valid,mri,bounds,centers

def unwrap_network(net):return net._orig_mod if hasattr(net,"_orig_mod") else net

def sample_feature_map(feat,local_zyx,patch_shape):
    if len(local_zyx)==0:return np.empty((0,int(feat.shape[1])),np.float32)
    ps=np.asarray(patch_shape,np.float32);loc=np.asarray(local_zyx,np.float32);norm=np.empty_like(loc)
    norm[:,0]=2*loc[:,2]/max(ps[2]-1,1)-1
    norm[:,1]=2*loc[:,1]/max(ps[1]-1,1)-1
    norm[:,2]=2*loc[:,0]/max(ps[0]-1,1)-1
    grid=torch.from_numpy(norm).to(feat.device)[None,:,None,None,:]
    sampled=F.grid_sample(feat.float(),grid,mode="bilinear",padding_mode="border",align_corners=True)[0,:,:,0,0].T
    return sampled.detach().cpu().numpy().astype(np.float32)

@torch.inference_mode()
def extract_encoder_nodes(predictor,data_np,centers):
    patch_shape=tuple(map(int,predictor.configuration_manager.patch_size))
    data_t=torch.from_numpy(data_np.astype(np.float32,copy=False))
    padded,revert=pad_nd_image(data_t,patch_shape,"constant",{"value":0},True,None)
    offset=np.asarray([revert[i].start for i in range(1,4)],np.float32)
    centers_pad=centers+offset[None,:]
    slicers=predictor._internal_get_sliding_window_slicers(tuple(padded.shape[1:]))
    net=unwrap_network(predictor.network).to(DEVICE).eval()
    gaussian=compute_gaussian(
        patch_shape,sigma_scale=1/8,value_scaling_factor=10,
        dtype=torch.float32,device=DEVICE
    ).detach().cpu().numpy().astype(np.float32)
    outputs=None;weights=np.zeros(len(centers),np.float32);stage_dims=None
    for sl in slicers:
        start=np.asarray([sl[1].start,sl[2].start,sl[3].start],np.float32)
        end=np.asarray([sl[1].stop,sl[2].stop,sl[3].stop],np.float32)
        ids=np.flatnonzero(np.all((centers_pad>=start[None,:])&(centers_pad<end[None,:]),axis=1))
        if len(ids)==0:continue
        patch=padded[sl][None].to(DEVICE,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.float16):
            skips=net.encoder(patch)
        selected=[skips[i] for i in ENCODER_STAGE_INDICES]
        if outputs is None:
            stage_dims=[int(v.shape[1]) for v in selected]
            outputs=np.zeros((len(centers),sum(stage_dims)),np.float32)
        local=centers_pad[ids]-start[None,:]
        obs=np.concatenate([sample_feature_map(v,local,patch_shape) for v in selected],axis=1)
        ii=np.rint(local).astype(np.int32)
        for d in range(3):ii[:,d]=np.clip(ii[:,d],0,patch_shape[d]-1)
        w=np.maximum(gaussian[ii[:,0],ii[:,1],ii[:,2]],1e-8)
        outputs[ids]+=obs*w[:,None];weights[ids]+=w
        del patch,skips,selected,obs
    assert outputs is not None and np.all(weights>0)
    outputs/=weights[:,None]
    return outputs,stage_dims

def handcrafted_features(data,pfg,stats,coords):
    gd,gh,gw=stats["pmax"].shape;zz,yy,xx=coords[:,0],coords[:,1],coords[:,2]
    feats=[
        stats["pmean"][zz,yy,xx],stats["pmax"][zz,yy,xx],stats["pstd"][zz,yy,xx],
        stats["ent_mean"][zz,yy,xx],stats["ent_max"][zz,yy,xx],stats["brain_frac"][zz,yy,xx],
    ]
    for mean,std,vmax in stats["modality"]:feats += [mean[zz,yy,xx],std[zz,yy,xx],vmax[zz,yy,xx]]
    grad=np.sqrt(sum(g.astype(np.float32)**2 for g in np.gradient(pfg.astype(np.float32))))
    gp,_,_=pad_to_cell(grad,CELL_SIZE,0);gb=block_view_3d(gp,CELL_SIZE)
    feats += [gb.mean(axis=(-3,-2,-1))[zz,yy,xx],gb.max(axis=(-3,-2,-1))[zz,yy,xx]]
    norm=(coords.astype(np.float32)+.5)/np.maximum(np.array([gd,gh,gw],np.float32),1)[None,:]
    feats += [norm[:,0],norm[:,1],norm[:,2]]
    return np.stack(feats,axis=1).astype(np.float32)

def compact_deep(deep,stage_dims,groups=SEMANTIC_DEEP_GROUPS):
    last=deep[:,-int(stage_dims[-1]):].astype(np.float32)
    return np.stack([last[:,idx].mean(1) for idx in np.array_split(np.arange(last.shape[1]),groups)],axis=1)

def build_edges(coords,hand,deep,stage_dims):
    n=len(coords);desc=np.concatenate([hand[:,:-3],compact_deep(deep,stage_dims)],axis=1).astype(np.float32)
    mu=desc.mean(0,keepdims=True);sd=desc.std(0,keepdims=True);sd[sd<1e-4]=1;desc=(desc-mu)/sd
    flags={}
    def add(i,j,k):
        key=(int(i),int(j))
        if key not in flags:flags[key]=[0,0,0,0]
        flags[key][k]=1
    lookup={tuple(c):i for i,c in enumerate(coords)}
    for i,c in enumerate(coords):
        for dz,dy,dx in [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]:
            j=lookup.get((int(c[0]+dz),int(c[1]+dy),int(c[2]+dx)))
            if j is not None:add(i,j,0)
    if n>1:
        _,idx=cKDTree(coords.astype(np.float32)).query(coords.astype(np.float32),k=min(SPATIAL_KNN+1,n))
        if np.ndim(idx)==1:idx=idx[:,None]
        for i in range(n):
            for j in np.atleast_1d(idx[i])[1:]:add(i,j,1);add(j,i,1)
        _,idx=cKDTree(desc).query(desc,k=min(FEATURE_KNN+1,n))
        if np.ndim(idx)==1:idx=idx[:,None]
        for i in range(n):
            for j in np.atleast_1d(idx[i])[1:]:add(i,j,2);add(j,i,2)
    for i in range(n):add(i,i,3)
    pairs=np.asarray(list(flags),np.int64);fl=np.asarray([flags[tuple(x)] for x in pairs],np.float32)
    src,dst=pairs[:,0],pairs[:,1]
    spatial=np.linalg.norm(coords[src].astype(np.float32)-coords[dst].astype(np.float32),axis=1)
    spatial/=max(float(np.linalg.norm(coords.max(0)-coords.min(0))),1.)
    fdist=np.linalg.norm(desc[src]-desc[dst],axis=1)/max(math.sqrt(desc.shape[1]),1.);fdist=np.clip(fdist,0,5)/5
    unit=desc/np.maximum(np.linalg.norm(desc,axis=1,keepdims=True),1e-8)
    cosine=np.sum(unit[src]*unit[dst],axis=1)
    edge_attr=np.concatenate([spatial[:,None],fdist[:,None],cosine[:,None],fl],axis=1).astype(np.float32)
    return pairs.T.astype(np.int32),edge_attr

print("Feature/graph helpers: READY")


In [ ]:
#@title 8. Stage5/6 persistence — SHA-verified legacy reuse + separate downstream root
_FEATURE_OUTER_CACHE={};_FEATURE_BANK_CACHE={};_FEATURE_COMMIT_CACHE={};_FEATURE_PROPS_CACHE={}
_DOWNSTREAM_OUTER_CACHE={}

def feature_outer_remote(fold):
    f=int(fold)
    if f not in _FEATURE_OUTER_CACHE:
        _FEATURE_OUTER_CACHE[f]=ensure_folder(FEATURE_REMOTE_OUTERS,f"outer_{f}")
    return _FEATURE_OUTER_CACHE[f]

def feature_remote(fold):
    f=int(fold)
    if f not in _FEATURE_BANK_CACHE:_FEATURE_BANK_CACHE[f]=ensure_folder(feature_outer_remote(f),"feature_bank")
    return _FEATURE_BANK_CACHE[f]

def feature_commit_remote(fold):
    f=int(fold)
    if f not in _FEATURE_COMMIT_CACHE:_FEATURE_COMMIT_CACHE[f]=ensure_folder(feature_outer_remote(f),"feature_commits")
    return _FEATURE_COMMIT_CACHE[f]

def props_remote(fold):
    f=int(fold)
    if f not in _FEATURE_PROPS_CACHE:_FEATURE_PROPS_CACHE[f]=ensure_folder(feature_outer_remote(f),"props")
    return _FEATURE_PROPS_CACHE[f]

def outer_remote(fold):
    f=int(fold)
    if f not in _DOWNSTREAM_OUTER_CACHE:_DOWNSTREAM_OUTER_CACHE[f]=ensure_folder(REMOTE_OUTERS,f"outer_{f}")
    return _DOWNSTREAM_OUTER_CACHE[f]

def fold_local(fold):
    d=FOLD_LOCAL/f"outer_{int(fold)}";d.mkdir(parents=True,exist_ok=True);return d

def feature_local(fold,case):
    d=fold_local(fold)/"features";d.mkdir(exist_ok=True);return d/f"{case}.npz"

def props_local(fold,case):
    d=fold_local(fold)/"props";d.mkdir(exist_ok=True);return d/f"{case}.pkl"

def feature_commit_name(case):return f"{case}.commit.json"

def valid_remote_feature(fold,case):
    cm=remote_file(feature_commit_remote(fold),feature_commit_name(case))
    fm=remote_file(feature_remote(fold),f"{case}.npz")
    pm=remote_file(props_remote(fold),f"{case}.pkl")
    if not (cm and fm and pm):
        return None

    cp=fold_local(fold)/f"{case}.commit.json"

    # Reuse a locally verified commit on rerun instead of downloading the same
    # tiny JSON repeatedly. The feature and props must still exist remotely.
    c=None
    if cp.exists():
        try:
            candidate=json.loads(cp.read_text())
            if (
                candidate.get("protocol_sha256")==LEGACY_FEATURE_PROTOCOL_SHA
                and candidate.get("cnn_checkpoint_sha256")==MODEL_META[int(fold)]["checkpoint_sha256"]
            ):
                c=candidate
        except Exception:
            c=None

    if c is None:
        cp.unlink(missing_ok=True)
        download(cm["id"],cp)
        c=json.loads(cp.read_text())

    if c.get("protocol_sha256")!=LEGACY_FEATURE_PROTOCOL_SHA:
        return None
    if c.get("cnn_checkpoint_sha256")!=MODEL_META[int(fold)]["checkpoint_sha256"]:
        return None

    return {"commit":c,"feature":fm,"props":pm}

def ensure_feature_local(fold,case):
    p=feature_local(fold,case)
    if p.exists():return p
    x=valid_remote_feature(fold,case)
    if not x:raise FileNotFoundError(("feature not built",fold,case))
    download(x["feature"]["id"],p)
    if sha256_file(p)!=x["commit"]["artifact_sha256"]:
        p.unlink(missing_ok=True);raise RuntimeError(("feature hash mismatch",fold,case))
    pp=props_local(fold,case)
    if not pp.exists():download(x["props"]["id"],pp)
    if sha256_file(pp)!=x["commit"]["props_sha256"]:
        pp.unlink(missing_ok=True);raise RuntimeError(("props hash mismatch",fold,case))
    return p


def stage_outer_feature_bank_local(fold,cases=None):
    """
    Stage every required Stage5/6 feature+props file onto /content BEFORE GAT.
    After this returns, Stage7/8 training reads no feature tensors from Drive.
    """
    fold=int(fold)
    cases=list(REG.case if cases is None else cases)
    staged_bytes=0

    for i,case in enumerate(cases,1):
        x=valid_remote_feature(fold,case)
        if not x:
            raise FileNotFoundError(("uncommitted Stage5/6 feature",fold,case))

        p=feature_local(fold,case)
        if not p.exists():
            download(x["feature"]["id"],p)

        if sha256_file(p)!=x["commit"]["artifact_sha256"]:
            p.unlink(missing_ok=True)
            download(x["feature"]["id"],p,force=True)
            if sha256_file(p)!=x["commit"]["artifact_sha256"]:
                raise RuntimeError(("feature SHA mismatch after restage",fold,case))

        pp=props_local(fold,case)
        if not pp.exists():
            download(x["props"]["id"],pp)

        if sha256_file(pp)!=x["commit"]["props_sha256"]:
            pp.unlink(missing_ok=True)
            download(x["props"]["id"],pp,force=True)
            if sha256_file(pp)!=x["commit"]["props_sha256"]:
                raise RuntimeError(("props SHA mismatch after restage",fold,case))

        staged_bytes += p.stat().st_size + pp.stat().st_size
        if i%10==0 or i==len(cases):
            print(
                "outer",fold,"local Stage5/6 staging",
                i,"/",len(cases),
                f"{staged_bytes/1024**3:.2f} GiB verified"
            )

    print("OUTER",fold,"LOCAL FEATURE STAGING PASS — training can run from /content")

print("Stage5/6 reuse bridge READY | frozen SHA",LEGACY_FEATURE_PROTOCOL_SHA)


In [ ]:
#@title 9. Build/reuse ONE outer fold's clean ResEncM-250 feature bank

def load_preprocessed(case):
    n=prep_npz(case)
    if not n.exists():
        x=remote_file(REMOTE_PREP_DATA,f"{case}.npz")
        if not x:raise FileNotFoundError(("preprocessed",case))
        download(x["id"],n)
    with np.load(n) as z:
        return z["data"].astype(np.float32),z["gt"].astype(np.uint8)

_GT_LRU=OrderedDict()
GT_LRU_MAX=3

def load_gt(case):
    case=str(case)

    if case in _GT_LRU:
        gt=_GT_LRU.pop(case)
        _GT_LRU[case]=gt
        return gt

    n=prep_npz(case)

    if not n.exists():
        x=remote_file(REMOTE_PREP_DATA,f"{case}.npz")
        if not x:
            raise FileNotFoundError(("preprocessed",case))
        download(x["id"],n)

    with np.load(n) as z:
        gt=z["gt"].astype(np.uint8)

    _GT_LRU[case]=gt

    while len(_GT_LRU)>GT_LRU_MAX:
        _GT_LRU.popitem(last=False)

    return gt


STAGE56_RUNTIME_IMPL_ID = (
    "v6-stage56-hardened-provenance-reuse-validate-20260919"
)


def repair_mislabeled_stage56_commits(fold):
    """
    Repair only Stage5/6 commit JSONs produced by the known v2 provenance-label bug.

    This function never rewrites feature NPZs or props PKLs, never runs ResEncM
    inference, and never fabricates artifact hashes. The existing strict validator
    and local SHA verification remain authoritative after this metadata repair.
    """
    fold=int(fold)
    already_correct=0
    repaired=0
    left_for_rebuild=0

    def _valid_sha256_hex(value):
        return (
            isinstance(value,str)
            and len(value)==64
            and all(ch in "0123456789abcdefABCDEF" for ch in value)
        )

    for case in REG.case:
        case=str(case)

        cm=remote_file(feature_commit_remote(fold),feature_commit_name(case))
        fm=remote_file(feature_remote(fold),f"{case}.npz")
        pm=remote_file(props_remote(fold),f"{case}.pkl")

        if not (cm and fm and pm):
            left_for_rebuild+=1
            continue

        cp=fold_local(fold)/f"{case}.commit.json"

        try:
            download(cm["id"],cp,force=True)
            commit=json.loads(cp.read_text())
        except Exception:
            left_for_rebuild+=1
            continue

        protocol=commit.get("protocol_sha256")

        if protocol==LEGACY_FEATURE_PROTOCOL_SHA:
            already_correct+=1
            continue

        if protocol!=PROTOCOL_SHA:
            left_for_rebuild+=1
            continue

        try:
            fold_matches=int(commit.get("outer_fold"))==fold
        except Exception:
            fold_matches=False

        known_bug_state=(
            commit.get("case")==case
            and fold_matches
            and commit.get("cnn_checkpoint_sha256")
                ==MODEL_META[fold]["checkpoint_sha256"]
            and _valid_sha256_hex(commit.get("artifact_sha256"))
            and _valid_sha256_hex(commit.get("props_sha256"))
            and commit.get("base_logits_dtype")=="float32"
            and commit.get("edge_attr_dtype")=="float32"
        )

        if not known_bug_state:
            left_for_rebuild+=1
            continue

        commit["protocol_sha256"]=LEGACY_FEATURE_PROTOCOL_SHA
        write_json(cp,commit)
        upload(
            cp,
            feature_commit_remote(fold),
            feature_commit_name(case),
            "application/json",
        )
        repaired+=1

    print("Stage5/6 commits already correct:",already_correct)
    print("Stage5/6 known-bug commits repaired:",repaired)
    print("Stage5/6 commits left for rebuild:",left_for_rebuild)

def validate_stage56_outer_complete(fold):
    fold=int(fold)
    bad=[]

    for case in map(str,REG.case.tolist()):
        x=valid_remote_feature(fold,case)
        if not x:
            bad.append(case)

    if bad:
        raise RuntimeError(
            (
                "Stage5/6 remote provenance validation failed",
                fold,
                len(bad),
                bad,
            )
        )

    print(
        "OUTER",
        fold,
        "REMOTE STAGE5/6 PROVENANCE PASS 93/93"
    )

validate_stage56_outer_complete._stage56_runtime_impl_id = (
    STAGE56_RUNTIME_IMPL_ID
)


def build_feature_bank_outer(fold):
    fold=int(fold)
    repair_mislabeled_stage56_commits(fold)

    assert _is_mount_alive(),"Drive mount unavailable before feature-bank verification"

    missing=[]
    for case in REG.case:
        if not valid_remote_feature(fold,str(case)):
            missing.append(str(case))

    print(
        "OUTER",
        fold,
        "Stage5/6 reusable:",
        len(REG)-len(missing),
        "/",
        len(REG),
        "| need build:",
        len(missing),
    )

    if not missing:
        print(
            "OUTER",
            fold,
            "RESENCM-250 FEATURE BANK COMPLETE 93/93 — ALL REUSED"
        )
        return

    seed_all(stable_seed("feature-bank",fold))
    predictor=load_resenc_predictor(fold)
    print("Building missing Stage5/6 for OUTER",fold,"from checkpoint",MODEL_META[fold]["checkpoint_sha256"][:16])

    for i,case in enumerate(missing,1):
        data,gt=load_preprocessed(case)
        with torch.inference_mode():
            logits=predictor.predict_logits_from_preprocessed_data(
                torch.from_numpy(data.astype(np.float32,copy=False))
            ).float().cpu()

        logits_np=logits.numpy().astype(np.float32)
        diff=(logits_np[1]-logits_np[0]).astype(np.float32)
        pfg=torch.softmax(logits,dim=0)[1].numpy().astype(np.float32)

        stats=cell_statistics(data,pfg,CELL_SIZE)
        coords=candidate_grid_coords(stats)
        node_logits,node_gt,node_valid,node_mri,bounds,centers=node_payload(diff,gt,data,coords)
        deep,stage_dims=extract_encoder_nodes(predictor,data,centers)
        hand=handcrafted_features(data,pfg,stats,coords)
        edge_index,edge_attr=build_edges(coords,hand,deep,stage_dims)

        # PRIORITY FIX:
        # Never quantize the baseline logits. These are the strongest completed CNN
        # prediction and are stored exactly as float32.
        base_store=logits_np.astype(np.float32,copy=False)

        p=feature_local(fold,case);tmp=Path(str(p)+".tmp.npz")
        np.savez_compressed(
            tmp,
            # Encoder path runs under FP16 mixed precision; storage remains FP16 for the
            # already-FP16 feature representation. Training reloads as float32.
            deep=deep.astype(np.float16),
            handcrafted=hand.astype(np.float32),
            grid_coords=coords.astype(np.int16),
            bounds=bounds.astype(np.int16),
            edge_index=edge_index.astype(np.int32),
            # 7-D edge attributes are cheap; keep full float32.
            edge_attr=edge_attr.astype(np.float32),
            node_voxel_logits=node_logits.astype(np.float32),
            node_voxel_gt=node_gt.astype(np.uint8),
            node_voxel_valid=node_valid.astype(np.uint8),
            node_voxel_mri=node_mri.astype(np.float16),
            base_logits_2ch=base_store,
            stage_dims=np.asarray(stage_dims,np.int16),
            preprocessed_shape=np.asarray(gt.shape,np.int32),
        )
        os.replace(tmp,p)

        shared_pp=prep_pkl(case)
        if not shared_pp.exists():
            x=remote_file(REMOTE_PREP_PROPS,f"{case}.pkl")
            download(x["id"],shared_pp)
        pp=props_local(fold,case)
        shutil.copy2(shared_pp,pp)

        commit={
            "case":case,
            "outer_fold":fold,
            "protocol_sha256":LEGACY_FEATURE_PROTOCOL_SHA,
            "cnn_checkpoint_sha256":MODEL_META[fold]["checkpoint_sha256"],
            "artifact_sha256":sha256_file(p),
            "props_sha256":sha256_file(pp),
            "node_count":int(len(coords)),
            "edge_count":int(edge_index.shape[1]),
            "deep_dim":int(deep.shape[1]),
            "handcrafted_dim":int(hand.shape[1]),
            "stage_dims":list(map(int,stage_dims)),
            "base_logits_dtype":"float32",
            "edge_attr_dtype":"float32",
            "created_utc":datetime.now(timezone.utc).isoformat(),
        }
        cp=fold_local(fold)/f"{case}.commit.json";write_json(cp,commit)
        upload(p,feature_remote(fold),mime="application/x-npz")
        upload(pp,props_remote(fold),mime="application/octet-stream")
        upload(cp,feature_commit_remote(fold),feature_commit_name(case),"application/json")

        if i%5==0 or i==len(missing):
            print(
                "outer",fold,
                "feature build",i,"/",len(missing),
                "| total valid after this build:",len(REG)-len(missing)+i,"/",len(REG),
                "nodes",len(coords),"dims",stage_dims
            )

    del predictor
    torch.cuda.empty_cache();gc.collect()
    print("OUTER",fold,"RESENCM-250 FEATURE BANK COMPLETE 93/93")

build_feature_bank_outer._stage56_runtime_impl_id = STAGE56_RUNTIME_IMPL_ID

print("Outer feature-bank builder: READY")


## Stage 7 → Stage 8 nesting lock

For every outer fold, Stage-7 tuning is trained on `train3` and validated on one `inner` fold.
Stage-8 tuning receives representations from that **tune GAT only**. The four-fold refit GAT is built
later and is used only for Stage-8 refit and the outer prediction. There is no identity-vs-GAT gate.


In [ ]:
#@title 10. Stage-7 GAT — FP32 + Dice+BCE + AdamW/cosine

def segment_softmax(scores,dst,num_nodes):
    E,H=scores.shape
    idx=dst[:,None].expand(-1,H)
    s=scores.float()
    mx=torch.full((num_nodes,H),-torch.inf,device=s.device,dtype=s.dtype)
    mx.scatter_reduce_(0,idx,s,reduce="amax",include_self=True)
    ex=torch.exp(s-mx[dst])
    den=torch.zeros((num_nodes,H),device=s.device,dtype=s.dtype)
    den.index_add_(0,dst,ex)
    return (ex/(den[dst]+1e-8)).to(scores.dtype)

class SparseEdgeGATLayer(nn.Module):
    def __init__(self,in_dim,out_dim,heads,edge_dim,dropout):
        super().__init__();assert out_dim%heads==0
        self.heads=heads;self.dk=out_dim//heads;self.dropout=dropout
        self.lin=nn.Linear(in_dim,out_dim,bias=False)
        self.a_src=nn.Parameter(torch.empty(heads,self.dk))
        self.a_dst=nn.Parameter(torch.empty(heads,self.dk))
        self.edge_bias=nn.Sequential(nn.Linear(edge_dim,heads),nn.Tanh())
        self.out=nn.Linear(out_dim,out_dim,bias=False)
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.a_src);nn.init.xavier_uniform_(self.a_dst)
        nn.init.zeros_(self.edge_bias[0].weight);nn.init.zeros_(self.edge_bias[0].bias)
        nn.init.xavier_uniform_(self.out.weight)

    def forward(self,x,edge_index,edge_attr):
        n=x.shape[0];src,dst=edge_index
        h=self.lin(x).view(n,self.heads,self.dk)
        e=(h[src]*self.a_src).sum(-1)+(h[dst]*self.a_dst).sum(-1)+self.edge_bias(edge_attr)
        a=segment_softmax(F.leaky_relu(e,.2),dst,n)
        a=F.dropout(a,p=self.dropout,training=self.training)
        msg=a[:,:,None]*h[src]
        out=msg.new_zeros((n,self.heads,self.dk))
        out.index_add_(0,dst,msg)
        return self.out(out.reshape(n,-1))

class TrueGAT(nn.Module):
    def __init__(self,in_dim,edge_dim=7):
        super().__init__()
        self.in_norm=nn.LayerNorm(in_dim)
        self.proj=nn.Sequential(
            nn.Linear(in_dim,GAT_HIDDEN),nn.GELU(),nn.Dropout(GUIDE_DROPOUT)
        )
        self.g1=SparseEdgeGATLayer(GAT_HIDDEN,GAT_HIDDEN,GAT_HEADS,edge_dim,GUIDE_DROPOUT)
        self.n1=nn.LayerNorm(GAT_HIDDEN)
        self.g2=SparseEdgeGATLayer(GAT_HIDDEN,GAT_HIDDEN,GAT_HEADS,edge_dim,GUIDE_DROPOUT)
        self.n2=nn.LayerNorm(GAT_HIDDEN)
        self.pool=nn.Linear(GAT_HIDDEN,1)
        self.vproj=nn.Sequential(
            nn.LayerNorm((CELL_SIZE**3)*4),
            nn.Linear((CELL_SIZE**3)*4,GAT_HIDDEN),
            nn.GELU(),nn.Dropout(GUIDE_DROPOUT)
        )

        fused_dim=GAT_HIDDEN*5
        self.node_head=nn.Linear(fused_dim+CELL_SIZE**3,CELL_SIZE**3)
        nn.init.zeros_(self.node_head.weight);nn.init.zeros_(self.node_head.bias)
        with torch.no_grad():
            self.node_head.weight[:,fused_dim:]=torch.eye(CELL_SIZE**3)

    def forward(self,x,edge_index,edge_attr,voxel_logits,voxel_mri):
        h0=self.proj(self.in_norm(x))
        h=self.n1(h0+F.dropout(F.gelu(self.g1(h0,edge_index,edge_attr)),
                              p=GUIDE_DROPOUT,training=self.training))
        h=self.n2(h+F.dropout(F.gelu(self.g2(h,edge_index,edge_attr)),
                             p=GUIDE_DROPOUT,training=self.training))
        w=torch.softmax(self.pool(h).squeeze(1),0)
        ca=(w[:,None]*h).sum(0);cm=h.mean(0)
        vc=torch.cat([voxel_logits,voxel_mri.reshape(voxel_mri.shape[0],-1)],1)
        vctx=self.vproj(vc)
        fused=torch.cat([h0,h,ca[None].expand_as(h),cm[None].expand_as(h),vctx],1)
        final_logits=self.node_head(torch.cat([fused,voxel_logits],1))
        return final_logits,h,vctx,torch.cat([ca,cm],0)

def soft_dice_loss(logits,target,valid,eps=1e-5):
    x=logits[valid];y=target[valid];p=torch.sigmoid(x)
    return 1-(2*(p*y).sum()+eps)/(p.sum()+y.sum()+eps)

def plain_bce_loss(logits,target,valid):
    return F.binary_cross_entropy_with_logits(logits[valid],target[valid])

def seg_loss(logits,target,valid):
    return 0.5*soft_dice_loss(logits,target,valid)+0.5*plain_bce_loss(logits,target,valid)

def make_guide_optimizer(model,schedule_horizon_epochs):
    opt=torch.optim.AdamW(
        model.parameters(),
        lr=GUIDE_LR,
        betas=GUIDE_BETAS,
        eps=GUIDE_EPS,
        weight_decay=GUIDE_WEIGHT_DECAY,
    )
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,T_max=max(1,int(schedule_horizon_epochs)),eta_min=GUIDE_MIN_LR
    )
    return opt,sch

print("Stage7 GAT + DiceBCE + AdamW/cosine READY")


In [ ]:
#@title 11. Feature loading / normalization / GAT case evaluation

_CASE_LRU=OrderedDict()
CASE_LRU_MAX=3

def load_feature_case(outer,case):
    key=(int(outer),case)
    if key in _CASE_LRU:
        v=_CASE_LRU.pop(key);_CASE_LRU[key]=v;return v
    p=ensure_feature_local(outer,case)
    with np.load(p) as z:
        a={k:np.asarray(z[k]) for k in z.files}
    _CASE_LRU[key]=a
    while len(_CASE_LRU)>CASE_LRU_MAX:
        _CASE_LRU.popitem(last=False)
    return a

def graph_x(a):
    return np.concatenate([
        a["deep"].astype(np.float32),
        a["handcrafted"].astype(np.float32)
    ],1)

def train_feature_norm(outer,cases):
    s=None;ss=None;n=0
    for c in cases:
        x=graph_x(load_feature_case(outer,c)).astype(np.float64)
        if s is None:
            s=np.zeros(x.shape[1],np.float64);ss=np.zeros_like(s)
        s+=x.sum(0);ss+=(x*x).sum(0);n+=len(x)
    mu=s/n
    var=np.maximum(ss/n-mu*mu,1e-8)
    return mu.astype(np.float32),np.sqrt(var).astype(np.float32)

def tensors_for_gat(outer,case,mu,sd):
    a=load_feature_case(outer,case)
    x=(graph_x(a)-mu[None,:])/np.maximum(sd[None,:],1e-6)
    return (
        torch.from_numpy(x).to(DEVICE),
        torch.from_numpy(a["edge_index"].astype(np.int64)).to(DEVICE),
        torch.from_numpy(a["edge_attr"].astype(np.float32)).to(DEVICE),
        torch.from_numpy(a["node_voxel_logits"].astype(np.float32)).to(DEVICE),
        torch.from_numpy(a["node_voxel_mri"].astype(np.float32)).to(DEVICE),
        torch.from_numpy(a["node_voxel_gt"].astype(np.float32)).to(DEVICE),
        torch.from_numpy(a["node_voxel_valid"].astype(bool)).to(DEVICE),
    )

def overwrite_node_logits(base_diff,node_logits,bounds):
    """Preserve the ResEncM baseline wherever the graph has no node coverage."""
    out=np.asarray(base_diff,np.float32).copy()
    for i,(z0,z1,y0,y1,x0,x1) in enumerate(bounds):
        dz,dy,dx=int(z1-z0),int(y1-y0),int(x1-x0)
        n=dz*dy*dx
        out[z0:z1,y0:y1,x0:x1]=node_logits[i,:n].reshape(dz,dy,dx)
    return out

@torch.inference_mode()
def gat_case_dice(model,outer,case,mu,sd):
    a=load_feature_case(outer,case)
    x,ei,ea,vl,vm,_,_=tensors_for_gat(outer,case,mu,sd)
    model.eval()
    final_node_logits,_,_,_=model(x,ei,ea,vl,vm)

    # IMPORTANT: epoch selection must evaluate the SAME complete-volume
    # semantics used later at inference. Uncovered voxels retain ResEncM,
    # rather than being silently replaced by a zero logit.
    base=a["base_logits_2ch"].astype(np.float32)
    base_diff=base[1]-base[0]
    diff=overwrite_node_logits(
        base_diff,
        final_node_logits.float().cpu().numpy(),
        a["bounds"],
    )

    gt=load_gt(case).astype(bool)
    pred=diff>0
    den=gt.sum()+pred.sum()
    return 1.0 if den==0 else float(2*np.logical_and(gt,pred).sum()/den)

print("GAT feature/metric helpers: READY — baseline-preserving selection")


In [ ]:
#@title 12. Tune + refit a REAL GAT for one outer fold — audited accumulation + resume

def outer_split(outer):
    outer=int(outer)
    inner=(outer+1)%5
    val=REG.loc[REG.fold==outer,"case"].tolist()
    inner_cases=REG.loc[REG.fold==inner,"case"].tolist()
    train3=REG.loc[~REG.fold.isin([outer,inner]),"case"].tolist()
    train4=REG.loc[REG.fold!=outer,"case"].tolist()
    return inner,train3,inner_cases,train4,val

def gat_remote(outer):return ensure_folder(outer_remote(outer),"gat")
def gat_tune_remote(outer):return ensure_folder(gat_remote(outer),"tune")
def gat_refit_remote(outer):return ensure_folder(gat_remote(outer),"refit")

def train_gat_epochs(outer,train_cases,mu,sd,epochs,tag,remote_parent,eval_cases=None):
    in_dim=graph_x(load_feature_case(outer,train_cases[0])).shape[1]
    seed=stable_seed("gat",tag,outer)
    seed_all(seed)

    model=TrueGAT(in_dim).to(DEVICE)
    opt,sch=make_guide_optimizer(model,GAT_MAX_TUNE_EPOCHS)
    history=[];best=None;no_improve=0;start_epoch=0

    # Exact restart from last durable evaluation boundary.
    last_meta=remote_file(remote_parent,"checkpoint_last.pth")
    if last_meta:
        last=fold_local(outer)/f"{tag}_resume.pth"
        last.unlink(missing_ok=True);download(last_meta["id"],last)
        ck=torch.load(last,map_location=DEVICE,weights_only=False)
        if ck.get("protocol_sha256")==DOWNSTREAM_PROTOCOL_SHA and ck.get("tag")==tag and ck.get("implementation_id")==STAGE7_IMPL_ID:
            model.load_state_dict(ck["model"])
            opt.load_state_dict(ck["optimizer"])
            sch.load_state_dict(ck["scheduler"])
            history=list(ck.get("history",[]))
            best=ck.get("best")
            no_improve=int(ck.get("no_improve",0))
            start_epoch=int(ck["epoch"])
            restore_rng(ck.get("rng"))
            print(tag,"RESUME epoch",start_epoch)

    for ep in range(start_epoch,epochs):
        order=train_cases.copy()
        random.Random(seed+ep).shuffle(order)
        model.train()
        ls=0.;case_count=0

        # Correct x8 accumulation, including the final partial group.
        for g0 in range(0,len(order),GAT_ACCUM_CASES):
            group=order[g0:g0+GAT_ACCUM_CASES]
            opt.zero_grad(set_to_none=True)

            for case in group:
                x,ei,ea,vl,vm,y,v=tensors_for_gat(outer,case,mu,sd)

                # Proven Stage-7 execution path: sparse GAT in full FP32.
                final_logits,_,_,_=model(x,ei,ea,vl,vm)
                raw_loss=seg_loss(final_logits,y,v)
                loss=raw_loss/len(group)
                loss.backward()

                ls+=float(raw_loss.detach().cpu())
                case_count+=1
                del x,ei,ea,vl,vm,y,v,final_logits,raw_loss,loss

            # Same stabilization used by the previously executed Stage-7 training.
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            opt.step()

        sch.step()
        rec={
            "epoch":ep+1,
            "train_loss":ls/max(1,case_count),
            "lr":float(opt.param_groups[0]["lr"])
        }
        history.append(rec)
        print(tag,rec)

        should_eval=eval_cases is not None and (
            (ep+1)%GAT_EVAL_EVERY==0 or ep==epochs-1
        )
        if should_eval:
            scores=[gat_case_dice(model,outer,c,mu,sd) for c in eval_cases]
            m=float(np.mean(scores))
            print(tag,"INNER Dice",m)
            if best is None or m>best["dice"]:
                best={"epoch":ep+1,"dice":m}
                bp=fold_local(outer)/f"{tag}_best.pth"
                torch.save({
                    "model":model.state_dict(),"mu":mu,"sd":sd,
                    "epoch":ep+1,"protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,"implementation_id":STAGE7_IMPL_ID,"dice":m,
                },bp)
                upload(bp,remote_parent,"checkpoint_best.pth")
                no_improve=0
            else:
                no_improve+=GAT_EVAL_EVERY

        # Recovery-only: persist restart state every completed epoch.
        # GAT validation/best-checkpoint selection remains on its original cadence.
        lp=fold_local(outer)/f"{tag}_last.pth"
        torch.save({
            "tag":tag,"model":model.state_dict(),
            "optimizer":opt.state_dict(),"scheduler":sch.state_dict(),
            "mu":mu,"sd":sd,
            "epoch":ep+1,"history":history,"best":best,
            "no_improve":no_improve,"rng":capture_rng(),
            "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,"implementation_id":STAGE7_IMPL_ID,
        },lp)
        upload(lp,remote_parent,"checkpoint_last.pth")

        if eval_cases is not None and no_improve>=GAT_EARLY_STOP_PATIENCE_EPOCHS:
            print(tag,"early stop after",no_improve,"epochs without improvement")
            break

    return model,history,best


# Optional engineering-only reuse:

# v3.4.1/v3.4.2 Stage7 is scientifically identical to v3.5.1 Stage7.
# Import only after exact protocol/implementation/epoch provenance checks.
V341_STAGE7_PROTOCOL_SHA="dee2a2ea4c4c158bcdd2f8c3fb4969d94aea33484c2e5efb9f7f482ad1fb6094"
V341_STAGE7_IMPL_ID="v3.4.1-gat-fp32-dice-bce-adamw-cosine-baseline-preserving-selection"
V341_STAGE7_ROOT_NAME="graphms_resencm250_true_hybrid_v3_4_1_dee2a2ea4c"

def try_import_v341_stage7(outer):
    outer=int(outer)
    root_meta=exact(IDS["nnunet_v2"],V341_STAGE7_ROOT_NAME,FOLDER_MIME,required=False)
    if not root_meta:
        return None
    try:
        old_outer_folds=folder(root_meta["id"],"outer_folds")
        old_outer=folder(old_outer_folds,f"outer_{outer}")
        old_gat=folder(old_outer,"gat")
        old_tune=folder(old_gat,"tune")
        old_refit=folder(old_gat,"refit")
    except FileNotFoundError:
        return None

    done_meta=remote_file(old_gat,"GAT_COMPLETE.json")
    tune_meta=remote_file(old_tune,"checkpoint_best.pth")
    refit_meta=remote_file(old_refit,"checkpoint_final.pth")
    if not (done_meta and tune_meta and refit_meta):
        return None

    idir=fold_local(outer)/"v341_stage7_import"
    idir.mkdir(parents=True,exist_ok=True)
    dp=idir/"GAT_COMPLETE.json";tp=idir/"checkpoint_best.pth";rp=idir/"checkpoint_final.pth"
    for p in [dp,tp,rp]:p.unlink(missing_ok=True)
    download(done_meta["id"],dp);download(tune_meta["id"],tp);download(refit_meta["id"],rp)

    old_done=json.loads(dp.read_text())
    tune_ck=torch.load(tp,map_location="cpu",weights_only=False)
    refit_ck=torch.load(rp,map_location="cpu",weights_only=False)
    checks=[
        old_done.get("protocol_sha256")==V341_STAGE7_PROTOCOL_SHA,
        old_done.get("implementation_id")==V341_STAGE7_IMPL_ID,
        tune_ck.get("protocol_sha256")==V341_STAGE7_PROTOCOL_SHA,
        tune_ck.get("implementation_id")==V341_STAGE7_IMPL_ID,
        refit_ck.get("protocol_sha256")==V341_STAGE7_PROTOCOL_SHA,
        refit_ck.get("implementation_id")==V341_STAGE7_IMPL_ID,
        int(refit_ck.get("epoch",-1))==int(old_done.get("selected_epoch",-2)),
    ]
    if not all(checks):
        print("v3.4.1 Stage7 exists but provenance does not match exactly; not importing it.")
        return None

    tune_new=dict(tune_ck);refit_new=dict(refit_ck)
    tune_new["protocol_sha256"]=DOWNSTREAM_PROTOCOL_SHA
    tune_new["implementation_id"]=STAGE7_IMPL_ID
    refit_new["protocol_sha256"]=DOWNSTREAM_PROTOCOL_SHA
    refit_new["implementation_id"]=STAGE7_IMPL_ID

    nt=fold_local(outer)/"gat_imported_v341_tune_best.pth"
    nr=fold_local(outer)/"gat_imported_v341_refit_final.pth"
    torch.save(tune_new,nt);torch.save(refit_new,nr)
    upload(nt,gat_tune_remote(outer),"checkpoint_best.pth")
    upload(nr,gat_refit_remote(outer),"checkpoint_final.pth")

    result={
        "outer":outer,
        "inner_fold":int(old_done["inner_fold"]),
        "selected_epoch":int(old_done["selected_epoch"]),
        "inner_best_dice":float(old_done["inner_best_dice"]),
        "imported_from_v341_stage7":True,
        "source_protocol_sha256":V341_STAGE7_PROTOCOL_SHA,
        "implementation_id":STAGE7_IMPL_ID,
        "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
    }
    outp=fold_local(outer)/"GAT_COMPLETE.json"
    write_json(outp,result);upload(outp,gat_remote(outer),"GAT_COMPLETE.json","application/json")
    print("outer",outer,"Stage7 imported from exact-matching v3.4.1/v3.4.2 checkpoint — no retraining")
    return result

# v3.3 Stage7 is scientifically compatible with v3.5.1 Stage7: both use
# baseline-preserving GAT epoch selection and the same Stage7 model/loss path.
# Import it ONLY when all old protocol/implementation checks pass.
V33_STAGE7_PROTOCOL_SHA="edd1bd77ba43030c6ceab29b6616bc56157400f5c143ed7e5ecca530115b2b6c"
V33_STAGE7_IMPL_ID="v3.3-gat-fp32-guide-dice-bce"
V33_STAGE7_ROOT_NAME="graphms_resencm250_true_hybrid_v3_3_edd1bd77ba"

def try_import_v33_stage7(outer):
    outer=int(outer)
    root_meta=exact(IDS["nnunet_v2"],V33_STAGE7_ROOT_NAME,FOLDER_MIME,required=False)
    if not root_meta:
        return None

    try:
        old_outer_folds=folder(root_meta["id"],"outer_folds")
        old_outer=folder(old_outer_folds,f"outer_{outer}")
        old_gat=folder(old_outer,"gat")
        old_tune=folder(old_gat,"tune")
        old_refit=folder(old_gat,"refit")
    except FileNotFoundError:
        return None

    done_meta=remote_file(old_gat,"GAT_COMPLETE.json")
    tune_meta=remote_file(old_tune,"checkpoint_best.pth")
    refit_meta=remote_file(old_refit,"checkpoint_final.pth")
    if not (done_meta and tune_meta and refit_meta):
        return None

    idir=fold_local(outer)/"v33_stage7_import"
    idir.mkdir(parents=True,exist_ok=True)
    dp=idir/"GAT_COMPLETE.json";tp=idir/"checkpoint_best.pth";rp=idir/"checkpoint_final.pth"
    for p in [dp,tp,rp]:
        p.unlink(missing_ok=True)

    download(done_meta["id"],dp)
    download(tune_meta["id"],tp)
    download(refit_meta["id"],rp)

    old_done=json.loads(dp.read_text())
    tune_ck=torch.load(tp,map_location="cpu",weights_only=False)
    refit_ck=torch.load(rp,map_location="cpu",weights_only=False)

    checks=[
        old_done.get("protocol_sha256")==V33_STAGE7_PROTOCOL_SHA,
        old_done.get("implementation_id")==V33_STAGE7_IMPL_ID,
        tune_ck.get("protocol_sha256")==V33_STAGE7_PROTOCOL_SHA,
        tune_ck.get("implementation_id")==V33_STAGE7_IMPL_ID,
        refit_ck.get("protocol_sha256")==V33_STAGE7_PROTOCOL_SHA,
        refit_ck.get("implementation_id")==V33_STAGE7_IMPL_ID,
        int(refit_ck.get("epoch",-1))==int(old_done.get("selected_epoch",-2)),
    ]
    if not all(checks):
        print("v3.3 Stage7 exists but provenance does not match exactly; training v3.5.1 Stage7.")
        return None

    # Rebind metadata only. Model weights / normalization / epoch are unchanged.
    tune_new=dict(tune_ck)
    tune_new["protocol_sha256"]=DOWNSTREAM_PROTOCOL_SHA
    tune_new["implementation_id"]=STAGE7_IMPL_ID
    refit_new=dict(refit_ck)
    refit_new["protocol_sha256"]=DOWNSTREAM_PROTOCOL_SHA
    refit_new["implementation_id"]=STAGE7_IMPL_ID

    nt=fold_local(outer)/"gat_imported_tune_best.pth"
    nr=fold_local(outer)/"gat_imported_refit_final.pth"
    torch.save(tune_new,nt);torch.save(refit_new,nr)
    upload(nt,gat_tune_remote(outer),"checkpoint_best.pth")
    upload(nr,gat_refit_remote(outer),"checkpoint_final.pth")

    result={
        "outer":outer,
        "inner_fold":int(old_done["inner_fold"]),
        "selected_epoch":int(old_done["selected_epoch"]),
        "inner_best_dice":float(old_done["inner_best_dice"]),
        "imported_from_v33_stage7":True,
        "source_protocol_sha256":V33_STAGE7_PROTOCOL_SHA,
        "implementation_id":STAGE7_IMPL_ID,
        "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
    }
    outp=fold_local(outer)/"GAT_COMPLETE.json"
    write_json(outp,result)
    upload(outp,gat_remote(outer),"GAT_COMPLETE.json","application/json")
    print("outer",outer,"Stage7 imported from exact-matching v3.3 checkpoint — no retraining")
    return result

def tune_refit_gat(outer):
    outer=int(outer)
    gr=gat_remote(outer)
    done=remote_json(gr,"GAT_COMPLETE.json",fold_local(outer))
    if (
        done
        and done.get("protocol_sha256")==DOWNSTREAM_PROTOCOL_SHA
        and done.get("implementation_id")==STAGE7_IMPL_ID
    ):
        print("GAT complete reuse outer",outer)
        return done

    imported=try_import_v341_stage7(outer)
    if imported is not None:
        return imported

    imported=try_import_v33_stage7(outer)
    if imported is not None:
        return imported

    inner,train3,inner_cases,train4,_=outer_split(outer)
    mu3,sd3=train_feature_norm(outer,train3)
    tm,history,best=train_gat_epochs(
        outer,train3,mu3,sd3,GAT_MAX_TUNE_EPOCHS,
        f"gat_tune_outer{outer}",gat_tune_remote(outer),inner_cases
    )
    if best is None:
        raise RuntimeError("No trained GAT checkpoint evaluated.")
    best_epoch=int(best["epoch"])
    del tm;torch.cuda.empty_cache();gc.collect()

    mu4,sd4=train_feature_norm(outer,train4)
    rm,_,_=train_gat_epochs(
        outer,train4,mu4,sd4,best_epoch,
        f"gat_refit_outer{outer}",gat_refit_remote(outer),eval_cases=None
    )
    final=fold_local(outer)/"gat_refit_final.pth"
    torch.save({
        "model":rm.state_dict(),"mu":mu4,"sd":sd4,
        "epoch":best_epoch,"protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,"implementation_id":STAGE7_IMPL_ID,
    },final)
    upload(final,gat_refit_remote(outer),"checkpoint_final.pth")
    del rm;torch.cuda.empty_cache();gc.collect()

    result={
        "outer":outer,"inner_fold":inner,
        "selected_epoch":best_epoch,
        "inner_best_dice":float(best["dice"]),
        "implementation_id":STAGE7_IMPL_ID,
        "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
    }
    p=fold_local(outer)/"GAT_COMPLETE.json"
    write_json(p,result)
    upload(p,gr,"GAT_COMPLETE.json","application/json")
    return result

print("GAT tune/refit routine: READY")


## Stage 8 / 9 — true spatial hybrid

The frozen refit GAT produces:

- local graph node embedding;
- local voxel-context embedding;
- global graph attention/mean descriptor.

Those are spatially scattered to the graph's 4×4×4 cell grid.

The four ResEncM-250 encoder scales remain separate.

For each 64³ voxel patch:

- CNN scale 1 is kept at the 16³ cell grid;
- CNN scale 2 is pooled to 8³;
- CNN scale 3 is pooled to 4³;
- CNN scale 4 is pooled to 2³;
- GAT features are pooled to matching levels;
- each level uses concatenation + SE;
- the deepest level uses self-attention;
- a top-down FPN reconstructs 16³;
- transposed convolutions reconstruct 32³ and 64³;
- local `[CNN logit | FLAIR | T1 | T2]` is used as decoder skip context;
- the final 1×1×1 residual layer starts at exact CNN identity.


In [ ]:
#@title 13. Phase-separated GAT caches — tune and refit are impossible to mix

def load_gat_phase(outer,phase):
    if phase=="tune": rr,name=gat_tune_remote(outer),"checkpoint_best.pth"
    elif phase=="refit": rr,name=gat_refit_remote(outer),"checkpoint_final.pth"
    else: raise ValueError(phase)
    m=remote_file(rr,name)
    if not m:raise FileNotFoundError(("GAT checkpoint",outer,phase))
    p=fold_local(outer)/f"gat_{phase}_reload.pth";p.unlink(missing_ok=True);download(m["id"],p)
    ck=torch.load(p,map_location="cpu",weights_only=False)
    if ck.get("protocol_sha256")!=DOWNSTREAM_PROTOCOL_SHA or ck.get("implementation_id")!=STAGE7_IMPL_ID:raise RuntimeError(("GAT protocol/implementation mismatch",outer,phase))
    in_dim=graph_x(load_feature_case(outer,REG.case.iloc[0])).shape[1]
    model=TrueGAT(in_dim).to(DEVICE);model.load_state_dict(ck["model"]);model.eval()
    for q in model.parameters():q.requires_grad_(False)
    return model,np.asarray(ck["mu"],np.float32),np.asarray(ck["sd"],np.float32),sha256_file(p)

def gat_cache_dir(outer,phase):
    d=fold_local(outer)/f"gat_cache_{phase}";d.mkdir(exist_ok=True);return d

def gat_cache_file(outer,phase,case):return gat_cache_dir(outer,phase)/f"{case}.npz"
def gat_cache_remote(outer,phase):return ensure_folder(gat_remote(outer),f"cache_{phase}")

def _gat_cache_valid(p,phase,cksha):
    try:
        with np.load(p) as z:
            req={"h","vctx","global_descriptor","gat_logits","phase","checkpoint_sha","protocol_sha","implementation_id"}
            return req<=set(z.files) and str(z["phase"].item())==phase and str(z["checkpoint_sha"].item())==cksha and str(z["protocol_sha"].item())==DOWNSTREAM_PROTOCOL_SHA and str(z["implementation_id"].item())==STAGE7_IMPL_ID
    except Exception:return False

@torch.inference_mode()
def ensure_gat_cache(outer,phase,cases):
    cases=list(dict.fromkeys(map(str,cases)))
    model,mu,sd,cksha=load_gat_phase(outer,phase);rd=gat_cache_remote(outer,phase)
    for i,case in enumerate(cases,1):
        p=gat_cache_file(outer,phase,case)
        if not _gat_cache_valid(p,phase,cksha):
            p.unlink(missing_ok=True);old=remote_file(rd,f"{case}.npz")
            if old:
                download(old["id"],p)
                if not _gat_cache_valid(p,phase,cksha):p.unlink(missing_ok=True)
        if not p.exists():
            x,ei,ea,vl,vm,_,_=tensors_for_gat(outer,case,mu,sd)
            # Frozen Stage-7 GAT cache inference is also full FP32.
            gat_logits,h,vctx,gd=model(x,ei,ea,vl,vm)
            tmp=Path(str(p)+".tmp.npz")
            np.savez_compressed(tmp,h=h.float().cpu().numpy().astype(np.float16),vctx=vctx.float().cpu().numpy().astype(np.float16),global_descriptor=gd.float().cpu().numpy().astype(np.float16),gat_logits=gat_logits.float().cpu().numpy().astype(np.float32),phase=np.asarray(phase),checkpoint_sha=np.asarray(cksha),protocol_sha=np.asarray(DOWNSTREAM_PROTOCOL_SHA),implementation_id=np.asarray(STAGE7_IMPL_ID))
            os.replace(tmp,p);upload(p,rd,f"{case}.npz","application/x-npz")
            del x,ei,ea,vl,vm,gat_logits,h,vctx,gd
        if i%10==0 or i==len(cases):print("outer",outer,phase,"GAT cache",i,"/",len(cases))
    del model;torch.cuda.empty_cache();gc.collect()

print("Phase-separated GAT cache routine READY")


In [ ]:
#@title 14. Stage8 exact fusion + Stage9 decoder

class SE3D(nn.Module):
    def __init__(self,c):
        super().__init__();h=max(8,c//8)
        self.net=nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Conv3d(c,h,1),nn.ReLU(inplace=True),
            nn.Conv3d(h,c,1),nn.Sigmoid()
        )
    def forward(self,x):return x*self.net(x)

class BottleneckSelfAttention(nn.Module):
    def __init__(self,c,heads=4):
        super().__init__()
        self.ln=nn.LayerNorm(c)
        self.attn=nn.MultiheadAttention(c,heads,batch_first=True)
    def forward(self,x):
        b,c,d,h,w=x.shape
        t=x.flatten(2).transpose(1,2)
        q=self.ln(t);a,_=self.attn(q,q,q,need_weights=False)
        return (t+a).transpose(1,2).reshape(b,c,d,h,w)

class ConvBlock(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv3d(cin,cout,3,padding=1,bias=False),
            nn.InstanceNorm3d(cout,affine=True),nn.GELU(),
            nn.Dropout3d(GUIDE_DROPOUT),
            nn.Conv3d(cout,cout,3,padding=1,bias=False),
            nn.InstanceNorm3d(cout,affine=True),nn.GELU(),
        )
    def forward(self,x):return self.net(x)

class GuideSpatialHybrid(nn.Module):
    def __init__(self,stage_dims):
        super().__init__()
        self.stage_dims=list(map(int,stage_dims));C=FPN_CHANNELS

        self.cproj=nn.ModuleList([nn.Conv3d(c,C,1) for c in self.stage_dims])
        self.gproj=nn.Conv3d(GAT_HIDDEN*2,C,1)
        self.glob=nn.Linear(GAT_HIDDEN*2,C)

        # Each scale uses CNN+GNN+global concatenation then SE.
        self.fuse=nn.ModuleList([
            nn.Sequential(
                nn.Conv3d(C*3,C,1),
                nn.GELU(),
                SE3D(C),
                ConvBlock(C,C)
            ) for _ in range(4)
        ])

        self.self_attn=BottleneckSelfAttention(C,4)
        self.lat3=ConvBlock(C,C);self.lat2=ConvBlock(C,C);self.lat1=ConvBlock(C,C)

        self.up32=nn.ConvTranspose3d(C,C//2,2,stride=2)
        self.ctx32=nn.Conv3d(1,C//2,1);self.dec32=ConvBlock(C,C//2)
        self.up64=nn.ConvTranspose3d(C//2,C//4,2,stride=2)
        self.ctx64=nn.Conv3d(1,C//4,1);self.dec64=ConvBlock(C//2,C//4)

        self.out=nn.Conv3d(C//4+1,1,1)
        nn.init.zeros_(self.out.weight);nn.init.zeros_(self.out.bias)
        with torch.no_grad():self.out.weight[0,-1,0,0,0]=1.0

    def forward(self,cnn_scales,h_grid,v_grid,global_desc,context64):
        g=self.gproj(torch.cat([h_grid,v_grid],1))
        gl=self.glob(global_desc).view(global_desc.shape[0],-1,1,1,1)

        pooled=[1,2,4,8]
        fs=[]
        for i,(cg,pool) in enumerate(zip(cnn_scales,pooled)):
            c=self.cproj[i](cg);gg=g
            if pool>1:
                c=F.avg_pool3d(c,pool,pool)
                gg=F.avg_pool3d(gg,pool,pool)
            ge=gl.expand(-1,-1,*c.shape[-3:])
            fs.append(self.fuse[i](torch.cat([c,gg,ge],1)))

        p4=self.self_attn(fs[3])
        p3=self.lat3(fs[2]+F.interpolate(p4,size=fs[2].shape[-3:],mode="trilinear",align_corners=False))
        p2=self.lat2(fs[1]+F.interpolate(p3,size=fs[1].shape[-3:],mode="trilinear",align_corners=False))
        p1=self.lat1(fs[0]+F.interpolate(p2,size=fs[0].shape[-3:],mode="trilinear",align_corners=False))

        y=self.up32(p1)
        y=self.dec32(torch.cat([y,self.ctx32(F.avg_pool3d(context64,2,2))],1))
        y=self.up64(y)
        y=self.dec64(torch.cat([y,self.ctx64(context64)],1))
        return self.out(torch.cat([y,context64[:,0:1]],1))

with torch.no_grad():
    dims=[128,256,320,320];B=1
    cs=[torch.randn(B,d,PATCH_CELLS,PATCH_CELLS,PATCH_CELLS,device=DEVICE) for d in dims]
    h=torch.randn(B,GAT_HIDDEN,PATCH_CELLS,PATCH_CELLS,PATCH_CELLS,device=DEVICE)
    v=torch.randn_like(h);gd=torch.randn(B,GAT_HIDDEN*2,device=DEVICE)
    ctx=torch.randn(B,1,PATCH_VOXELS,PATCH_VOXELS,PATCH_VOXELS,device=DEVICE)
    m=GuideSpatialHybrid(dims).to(DEVICE);o=m(cs,h,v,gd,ctx)
    assert o.shape==ctx.shape
    assert torch.equal(o,ctx)
del m,o,cs,h,v,gd,ctx
torch.cuda.empty_cache()

print("Fixed Stage8/9 smoke test PASS")


In [ ]:
#@title 15. Patch construction — phase-safe GAT + CNN-logit decoder skip

_GAT_LRU=OrderedDict()
GAT_LRU_MAX=3

def load_gat_cache(outer,phase,case):
    p=gat_cache_file(outer,phase,case)

    if not p.exists():
        raise FileNotFoundError(("GAT cache",outer,phase,case))

    st=p.stat()

    key=(
        int(outer),
        str(phase),
        str(case),
        int(st.st_mtime_ns),
        int(st.st_size),
    )

    if key in _GAT_LRU:
        g=_GAT_LRU.pop(key)
        _GAT_LRU[key]=g
        return g

    with np.load(p) as z:
        g={k:np.asarray(z[k]) for k in z.files}

    # Remove an old in-RAM version if this exact case/phase
    # has been regenerated on disk.
    for k in list(_GAT_LRU.keys()):
        if k[:3]==key[:3]:
            _GAT_LRU.pop(k,None)

    _GAT_LRU[key]=g

    while len(_GAT_LRU)>GAT_LRU_MAX:
        _GAT_LRU.popitem(last=False)

    return g

def crop_pad(arr,start,size):
    out=np.zeros(arr.shape[:-3]+(size,size,size),dtype=arr.dtype)
    z0,y0,x0=map(int,start);Z,Y,X=arr.shape[-3:]
    zs0=max(z0,0);ys0=max(y0,0);xs0=max(x0,0);zs1=min(z0+size,Z);ys1=min(y0+size,Y);xs1=min(x0+size,X)
    if zs1<=zs0 or ys1<=ys0 or xs1<=xs0:return out
    oz0=zs0-z0;oy0=ys0-y0;ox0=xs0-x0
    out[...,oz0:oz0+zs1-zs0,oy0:oy0+ys1-ys0,ox0:ox0+xs1-xs0]=arr[...,zs0:zs1,ys0:ys1,xs0:xs1]
    return out

def choose_patch_start(gt,rng):
    shape=np.asarray(gt.shape,int)
    if gt.any() and rng.random()<0.6:
        pts=np.argwhere(gt);center=pts[int(rng.integers(0,len(pts)))];start=center-PATCH_VOXELS//2
    else:start=np.array([int(rng.integers(-PATCH_VOXELS//4,max(1,shape[d]-3*PATCH_VOXELS//4+1))) for d in range(3)])
    return (np.floor(start/CELL_SIZE)*CELL_SIZE).astype(int)

def dense_node_patch(a,g,start):
    coords=a["grid_coords"].astype(np.int32);cstart=np.floor(np.asarray(start)/CELL_SIZE).astype(int);rel=coords-cstart[None,:]
    ids=np.flatnonzero(np.all((rel>=0)&(rel<PATCH_CELLS),axis=1));pos=rel[ids]
    stage_dims=list(map(int,a["stage_dims"]));deep=a["deep"].astype(np.float32);scales=[];off=0
    for d in stage_dims:
        grid=np.zeros((d,PATCH_CELLS,PATCH_CELLS,PATCH_CELLS),np.float32)
        if len(ids):grid[:,pos[:,0],pos[:,1],pos[:,2]]=deep[ids,off:off+d].T
        scales.append(grid);off+=d
    hg=np.zeros((GAT_HIDDEN,PATCH_CELLS,PATCH_CELLS,PATCH_CELLS),np.float32);vg=np.zeros_like(hg)
    if len(ids):
        hg[:,pos[:,0],pos[:,1],pos[:,2]]=g["h"].astype(np.float32)[ids].T
        vg[:,pos[:,0],pos[:,1],pos[:,2]]=g["vctx"].astype(np.float32)[ids].T
    return scales,hg,vg,g["global_descriptor"].astype(np.float32)

def make_patch(outer,case,start,gat_phase,gt=None):
    a=load_feature_case(outer,case);g=load_gat_cache(outer,gat_phase,case)

    if gt is None:
        gt=load_gt(case)
    base=a["base_logits_2ch"].astype(np.float32);diff=(base[1]-base[0]).astype(np.float32)
    scales,h,v,gd=dense_node_patch(a,g,start);valid=np.ones(gt.shape,np.uint8)
    return {"scales":scales,"h":h,"v":v,"gd":gd,"context":crop_pad(diff[None],start,PATCH_VOXELS),"target":crop_pad(gt[None].astype(np.float32),start,PATCH_VOXELS),"valid":crop_pad(valid[None],start,PATCH_VOXELS).astype(bool)}

def batch_patches(outer,cases,seed,gat_phase):
    rng=np.random.default_rng(seed);items=[]
    for _ in range(BATCH_SIZE):
        case=str(cases[int(rng.integers(0,len(cases)))])
        gt=load_gt(case)

        items.append(
            make_patch(
                outer,
                case,
                choose_patch_start(gt,rng),
                gat_phase,
                gt=gt,
            )
        )
    nd=len(items[0]["scales"]);scales=[torch.from_numpy(np.stack([x["scales"][i] for x in items])).to(DEVICE) for i in range(nd)]
    return scales,torch.from_numpy(np.stack([x["h"] for x in items])).to(DEVICE),torch.from_numpy(np.stack([x["v"] for x in items])).to(DEVICE),torch.from_numpy(np.stack([x["gd"] for x in items])).to(DEVICE),torch.from_numpy(np.stack([x["context"] for x in items])).to(DEVICE),torch.from_numpy(np.stack([x["target"] for x in items])).to(DEVICE),torch.from_numpy(np.stack([x["valid"] for x in items])).to(DEVICE)

print("Phase-safe patch builder READY")


In [ ]:
#@title 16. Stage8/9 direct training + inference — no search

def fusion_remote(outer):return ensure_folder(outer_remote(outer),"fusion")
def fusion_tune_remote(outer):return ensure_folder(fusion_remote(outer),"tune")
def fusion_refit_remote(outer):return ensure_folder(fusion_remote(outer),"refit")

def stage_dims_outer(outer):
    dims=list(map(int,load_feature_case(outer,REG.case.iloc[0])["stage_dims"]))
    for c in REG.case.iloc[1:6]:
        if list(map(int,load_feature_case(outer,c)["stage_dims"]))!=dims:
            raise RuntimeError("stage dim mismatch")
    return dims

@torch.inference_mode()
def predict_preprocessed_hybrid(model,outer,case,gat_phase,stride=32):
    gt=load_gt(case)
    shape=np.asarray(gt.shape,int);target=np.maximum(shape,PATCH_VOXELS);starts=[]
    for d in range(3):
        mx=max(0,int(target[d]-PATCH_VOXELS))
        ss=list(range(0,mx+1,stride)) or [0]
        if ss[-1]!=mx:ss.append(mx)
        starts.append(ss)

    accum=np.zeros(tuple(target),np.float32)
    count=np.zeros(tuple(target),np.float32)
    model.eval()

    for z in starts[0]:
      for y in starts[1]:
       for x in starts[2]:
        item=make_patch(
            outer,
            case,
            (z,y,x),
            gat_phase,
            gt=gt,
        )
        scales=[torch.from_numpy(vv[None]).to(DEVICE) for vv in item["scales"]]
        h=torch.from_numpy(item["h"][None]).to(DEVICE)
        v=torch.from_numpy(item["v"][None]).to(DEVICE)
        gd=torch.from_numpy(item["gd"][None]).to(DEVICE)
        ctx=torch.from_numpy(item["context"][None]).to(DEVICE)

        with torch.autocast("cuda",dtype=torch.float16):
            final_logit=model(scales,h,v,gd,ctx)[0,0].float().cpu().numpy()

        accum[z:z+PATCH_VOXELS,y:y+PATCH_VOXELS,x:x+PATCH_VOXELS]+=final_logit
        count[z:z+PATCH_VOXELS,y:y+PATCH_VOXELS,x:x+PATCH_VOXELS]+=1

    return (accum/np.maximum(count,1e-6))[:shape[0],:shape[1],:shape[2]],gt

def pre_dice_from_diff(diff,gt):
    p=diff>0;g=gt.astype(bool);den=p.sum()+g.sum()
    return 1.0 if den==0 else float(2*np.logical_and(p,g).sum()/den)

def train_fusion(outer,train_cases,val_cases,epochs,remote_parent,tag,gat_phase):
    assert gat_phase in {"tune","refit"}
    dims=stage_dims_outer(outer)
    seed=stable_seed("fusion",tag,outer,gat_phase);seed_all(seed)

    model=GuideSpatialHybrid(dims).to(DEVICE)

    # Tune and refit share the same 60-epoch cosine schedule definition.
    # If epoch e is selected, refit executes the same first e LR steps.
    opt,sch=make_guide_optimizer(model,FUSION_MAX_TUNE_EPOCHS)
    scaler=torch.amp.GradScaler("cuda")

    best=None;no_improve=0;history=[];start_epoch=0
    last_meta=remote_file(remote_parent,"checkpoint_last.pth")

    if last_meta:
        last=fold_local(outer)/f"{tag}_resume.pth"
        last.unlink(missing_ok=True);download(last_meta["id"],last)
        ck=torch.load(last,map_location=DEVICE,weights_only=False)

        if (
            ck.get("protocol_sha256")==DOWNSTREAM_PROTOCOL_SHA
            and ck.get("implementation_id")==FUSION_IMPL_ID
            and ck.get("tag")==tag
            and ck.get("gat_phase")==gat_phase
        ):
            model.load_state_dict(ck["model"])
            opt.load_state_dict(ck["optimizer"]);sch.load_state_dict(ck["scheduler"])
            scaler.load_state_dict(ck["scaler"])
            history=list(ck.get("history",[]));best=ck.get("best")
            no_improve=int(ck.get("no_improve",0));start_epoch=int(ck["epoch"])
            restore_rng(ck.get("rng"))
            print(tag,"RESUME",start_epoch,gat_phase)

    for ep in range(start_epoch,epochs):
        model.train();opt.zero_grad(set_to_none=True);loss_sum=0.0

        for it in range(FUSION_ITERS_PER_EPOCH):
            scales,h,v,gd,ctx,y,valid=batch_patches(
                outer,train_cases,
                stable_seed("fusion-patch",tag,outer,gat_phase,ep,it),
                gat_phase,
            )
            # Exact normalization for full AND final partial accumulation groups.
            group_start=(it//GRAD_ACCUM_STEPS)*GRAD_ACCUM_STEPS
            group_size=min(GRAD_ACCUM_STEPS,FUSION_ITERS_PER_EPOCH-group_start)

            with torch.autocast("cuda",dtype=torch.float16):
                final_logits=model(scales,h,v,gd,ctx)
                raw_loss=seg_loss(final_logits,y,valid)
                loss=raw_loss/float(group_size)

            scaler.scale(loss).backward()

            if (it+1)%GRAD_ACCUM_STEPS==0 or it+1==FUSION_ITERS_PER_EPOCH:
                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)

            loss_sum+=float(raw_loss.detach().cpu())

        sch.step()
        rec={
            "epoch":ep+1,
            "train_loss":loss_sum/FUSION_ITERS_PER_EPOCH,
            "lr":float(opt.param_groups[0]["lr"]),
            "gat_phase":gat_phase,
        }
        history.append(rec);print(tag,rec)

        if val_cases is not None and ((ep+1)%FUSION_EVAL_EVERY==0 or ep==epochs-1):
            scores=[
                pre_dice_from_diff(*predict_preprocessed_hybrid(model,outer,c,gat_phase))
                for c in val_cases
            ]
            mean_dice=float(np.mean(scores));print(tag,"INNER Dice",mean_dice)

            if best is None or mean_dice>best["dice"]:
                best={"epoch":ep+1,"dice":mean_dice}
                bp=fold_local(outer)/f"{tag}_best.pth"
                torch.save({
                    "model":model.state_dict(),"epoch":ep+1,"dice":mean_dice,
                    "stage_dims":dims,"gat_phase":gat_phase,
                    "implementation_id":FUSION_IMPL_ID,
                    "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
                },bp)
                upload(bp,remote_parent,"checkpoint_best.pth");no_improve=0
            else:
                no_improve+=FUSION_EVAL_EVERY

        # Recovery-only: persist restart state every completed epoch.
        # Validation and best-checkpoint selection remain on the original 5-epoch cadence.
        lp=fold_local(outer)/f"{tag}_last.pth"
        torch.save({
            "tag":tag,"model":model.state_dict(),
            "optimizer":opt.state_dict(),"scheduler":sch.state_dict(),
            "scaler":scaler.state_dict(),"epoch":ep+1,
            "history":history,"best":best,"no_improve":no_improve,
            "stage_dims":dims,"gat_phase":gat_phase,
            "rng":capture_rng(),
            "implementation_id":FUSION_IMPL_ID,
            "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
        },lp)
        upload(lp,remote_parent,"checkpoint_last.pth")

        if val_cases is not None and no_improve>=FUSION_EARLY_STOP_PATIENCE_EPOCHS:
            print(tag,"early stop",no_improve);break

    return model,history,best

print("Single-path Stage8/9 training READY")


In [ ]:
#@title 17. Stage8/9 tune -> refit — one exact path

def tune_refit_fusion(outer):
    outer=int(outer)
    fr=fusion_remote(outer)
    done=remote_json(fr,"FUSION_COMPLETE.json",fold_local(outer))

    if (
        done
        and done.get("protocol_sha256")==DOWNSTREAM_PROTOCOL_SHA
        and done.get("implementation_id")==FUSION_IMPL_ID
    ):
        return done

    inner,train3,inner_cases,train4,outer_cases=outer_split(outer)
    assert not(set(train3)&set(inner_cases))
    assert not(set(train4)&set(outer_cases))
    assert set(train3)|set(inner_cases)==set(train4)

    ensure_gat_cache(outer,"tune",train3+inner_cases)

    tm,_,best=train_fusion(
        outer,train3,inner_cases,
        FUSION_MAX_TUNE_EPOCHS,
        fusion_tune_remote(outer),
        f"fusion_tune_outer{outer}",
        "tune",
    )
    if best is None:raise RuntimeError("No fusion checkpoint evaluated")

    selected_epoch=int(best["epoch"])
    del tm;torch.cuda.empty_cache();gc.collect()

    ensure_gat_cache(outer,"refit",train4)

    rm,_,_=train_fusion(
        outer,train4,None,
        selected_epoch,
        fusion_refit_remote(outer),
        f"fusion_refit_outer{outer}",
        "refit",
    )

    p=fold_local(outer)/"fusion_refit_final.pth"
    torch.save({
        "model":rm.state_dict(),
        "epoch":selected_epoch,
        "stage_dims":stage_dims_outer(outer),
        "gat_phase":"refit",
        "implementation_id":FUSION_IMPL_ID,
        "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
    },p)
    upload(p,fusion_refit_remote(outer),"checkpoint_final.pth")
    del rm;torch.cuda.empty_cache();gc.collect()

    result={
        "outer":outer,
        "inner_fold":inner,
        "selected_epoch":selected_epoch,
        "inner_best_dice":float(best["dice"]),
        "outer_gt_used_before_prediction":False,
        "implementation_id":FUSION_IMPL_ID,
        "protocol_sha256":DOWNSTREAM_PROTOCOL_SHA,
    }
    q=fold_local(outer)/"FUSION_COMPLETE.json"
    write_json(q,result);upload(q,fr,"FUSION_COMPLETE.json","application/json")
    return result

def load_fusion_refit(outer):
    m=remote_file(fusion_refit_remote(outer),"checkpoint_final.pth")
    if not m:raise FileNotFoundError(("fusion final",outer))

    p=fold_local(outer)/"fusion_refit_reload.pth"
    p.unlink(missing_ok=True);download(m["id"],p)
    ck=torch.load(p,map_location=DEVICE,weights_only=False)

    if (
        ck.get("protocol_sha256")!=DOWNSTREAM_PROTOCOL_SHA
        or ck.get("implementation_id")!=FUSION_IMPL_ID
        or ck.get("gat_phase")!="refit"
    ):
        raise RuntimeError("final fusion provenance mismatch")

    model=GuideSpatialHybrid(ck["stage_dims"]).to(DEVICE)
    model.load_state_dict(ck["model"]);model.eval()
    return model

print("Single-path nested Stage8 tune/refit READY")


In [ ]:
#@title 18. Original geometry + validated guide-contained Stage11 filter5

PLANS=json.loads((MODEL_LOCAL/"plans.json").read_text())
DATASET_JSON=json.loads((MODEL_LOCAL/"dataset.json").read_text())
PM=PlansManager(PLANS)
CM=PM.get_configuration("3d_fullres")
LM=PM.get_label_manager(DATASET_JSON)

GT_ORIG=LOCAL/"gt_original";GT_ORIG.mkdir(exist_ok=True)

def ensure_orig_gt(case):
    p=GT_ORIG/f"{case}.nii.gz"
    if not p.exists():download(RMAP[case].gt_id,p)
    return p

def convert_original(outer,case,logits):
    pp=props_local(outer,case)
    if not pp.exists():
        x=valid_remote_feature(outer,case);download(x["props"]["id"],pp)
    with pp.open("rb") as f:props=pickle.load(f)

    return convert_predicted_logits_to_segmentation_with_correct_shape(
        torch.from_numpy(np.asarray(logits,np.float32)),
        PM,CM,LM,props,
        return_probabilities=True,
        num_threads_torch=2,
    )

def validated_stage11_filter5(prob):
    """
    Guide-contained Stage11 using the previously validated project setting:
      p >= 0.5
      -> 3-D 26-connected components
      -> remove components with <5 voxels (filter5).

    Numeric threshold/component size are PROJECT-VALIDATED settings, not claimed
    as mandatory numbers supplied by the guide. No new sweep is performed.
    """
    mask=np.asarray(prob,np.float32)>=GUIDE_THRESHOLD
    if not mask.any():return mask

    lab,n=ndi.label(mask,structure=ndi.generate_binary_structure(3,3))
    if n==0:return mask

    sizes=np.bincount(lab.ravel())
    keep=sizes>=GUIDE_MIN_COMPONENT_VOXELS
    keep[0]=False
    return keep[lab]

print("Stage11 READY: validated p=0.5 + 26-connectivity + filter5; NO new search")


In [ ]:
#@title 19. Final prediction — uncovered GAT voxels preserve ResEncM baseline

def predictions_remote(outer):return ensure_folder(outer_remote(outer),"outer_predictions")

def overwrite_nodes_on_baseline(base_diff,node_logits,bounds):
    out=np.asarray(base_diff,np.float32).copy()
    for i,(z0,z1,y0,y1,x0,x1) in enumerate(bounds):
        dz,dy,dx=int(z1-z0),int(y1-y0),int(x1-x0);n=dz*dy*dx;out[z0:z1,y0:y1,x0:x1]=node_logits[i,:n].reshape(dz,dy,dx)
    return out

@torch.inference_mode()
def gat_diff_full(outer,case):
    a=load_feature_case(outer,case);base=a["base_logits_2ch"].astype(np.float32);base_diff=(base[1]-base[0]).astype(np.float32)
    ensure_gat_cache(outer,"refit",[case])
    with np.load(gat_cache_file(outer,"refit",case)) as z:node_logits=z["gat_logits"].astype(np.float32)
    return overwrite_nodes_on_baseline(base_diff,node_logits,a["bounds"])

def make_twoch(base,diff):
    out=base.astype(np.float32).copy();out[1]=out[0]+diff.astype(np.float32);return out

def predict_outer_fold(outer):
    _,_,_,_,val=outer_split(outer);ensure_gat_cache(outer,"refit",val);hybrid=load_fusion_refit(outer);rr=predictions_remote(outer);rows=[]
    for i,case in enumerate(val,1):
        old=(
            remote_file(rr,f"{case}.outer_pred.npz")
            or remote_file(rr,f"{case}.npz")
        )
        local=fold_local(outer)/f"{case}.outer_pred.npz"
        valid_old=False
        if old:
            local.unlink(missing_ok=True);download(old["id"],local)
            try:
                with np.load(local) as z:
                    valid_old=(
                        "protocol_sha" in z.files
                        and str(z["protocol_sha"].item())==DOWNSTREAM_PROTOCOL_SHA
                        and "implementation_id" in z.files
                        and str(z["implementation_id"].item())==FUSION_IMPL_ID
                        and all(k in z.files for k in ["cnn","gat","hybrid","spacing_xyz"])
                    )
            except Exception:
                valid_old=False
            if not valid_old:
                local.unlink(missing_ok=True)
        if not valid_old:
            a=load_feature_case(outer,case);base=a["base_logits_2ch"].astype(np.float32);cnn_diff=base[1]-base[0];gat_diff=gat_diff_full(outer,case);hyb_diff,_=predict_preprocessed_hybrid(hybrid,outer,case,"refit")
            _,cp=convert_original(outer,case,make_twoch(base,cnn_diff));_,gp=convert_original(outer,case,make_twoch(base,gat_diff));_,hp=convert_original(outer,case,make_twoch(base,hyb_diff))
            im=sitk.ReadImage(str(ensure_orig_gt(case)));gt=sitk.GetArrayFromImage(im)>0;assert cp.shape[1:]==gt.shape and gp.shape[1:]==gt.shape and hp.shape[1:]==gt.shape
            tmp=Path(str(local)+".tmp.npz");np.savez_compressed(tmp,cnn=cp[1].astype(np.float32),gat=gp[1].astype(np.float32),hybrid=hp[1].astype(np.float32),spacing_xyz=np.asarray(im.GetSpacing(),np.float32),protocol_sha=np.asarray(DOWNSTREAM_PROTOCOL_SHA),implementation_id=np.asarray(FUSION_IMPL_ID));os.replace(tmp,local);upload(local,rr,mime="application/x-npz")
        rows.append({"outer":outer,"case":case});print("outer",outer,"prediction",i,"/",len(val),case)
    del hybrid;torch.cuda.empty_cache();gc.collect();return rows

print("Outer prediction routine READY")


In [ ]:
#@title 20. v3.5.1 FINAL GUIDE-LEAN AUDITED PRE-RUN LOCK + execute

rows=[]
for outer in range(5):
    inner,train3,inner_cases,train4,outer_cases=outer_split(outer)
    ok=(
        not(set(train3)&set(inner_cases))
        and not(set(train4)&set(outer_cases))
        and set(train3)|set(inner_cases)==set(train4)
    )
    rows.append({
        "outer":outer,"inner":inner,
        "train3":len(train3),"inner_cases":len(inner_cases),"train4":len(train4),
        "outer_cases":len(outer_cases),"disjoint":ok,
    })

AUDIT=pd.DataFrame(rows)
display(AUDIT)
assert AUDIT.disjoint.all()

PIPELINE_LOCK=pd.DataFrame([
    ["Stage7","GAT / message passing","FIXED; exact compatible checkpoints reusable"],
    ["Stage8","concat + SE + self-attention + multi-scale","ONE PATH"],
    ["Stage9","transposed conv + skip + final 1x1x1","FIXED"],
    ["Stage10","Dice+BCE","FIXED"],
    ["Stage11","p=0.5 + 26-connected filter5","VALIDATED PROJECT SETTING; NO SWEEP"],
    ["Stage15","AdamW + cosine + dropout + weight decay","FIXED"],
    ["Fusion epoch","250 iterations","LEAN"],
    ["Fusion max","60 epochs","CEILING PRESERVED"],
    ["Stage7 early stop","20 epochs without new best inner Dice","AUDITED STAGE7 VALUE"],
    ["Fusion early stop","20 epochs without new best inner Dice","METRIC-FIRST VALUE"],
    ["Fusion checkpoint","every completed epoch","RECOVERY ONLY"],
    ["Partial grad accumulation","normalize by actual group size","CORRECT"],
    ["Fusion/loss/optimizer/scheduler search","none","REMOVED"],
    ["Stage11 parameter sweep","none","REMOVED"],
    ["Label smoothing / stochastic depth / AMSGrad / warmup","none","REMOVED"],
    ["Stage2/3 literal BET/N4/ANTs + every listed augmentation","frozen upstream nnU-Net lineage","NOT FALSELY CLAIMED"],
    ["Stage12-14","after segmentation freeze","NOT RUN IN THIS NOTEBOOK"],
],columns=["Area","Pipeline","Status"])
display(PIPELINE_LOCK)

print("V3.5.1 FINAL GUIDE-LEAN AUDITED PRE-RUN LOCK PASS")
print("Frozen Stage5/6 SHA:",LEGACY_FEATURE_PROTOCOL_SHA)
print("Downstream v3.5.1 SHA:",DOWNSTREAM_PROTOCOL_SHA)
print("Fusion iterations/epoch:",FUSION_ITERS_PER_EPOCH)
print("Fusion max epochs:",FUSION_MAX_TUNE_EPOCHS)
print("Stage7 early-stop patience:",GAT_EARLY_STOP_PATIENCE_EPOCHS)
print("Fusion early-stop patience:",FUSION_EARLY_STOP_PATIENCE_EPOCHS)
print("Stage11: p=",GUIDE_THRESHOLD," minvox=",GUIDE_MIN_COMPONENT_VOXELS," connectivity=",GUIDE_CONNECTIVITY,sep="")

print("Checking exact Stage5/6 runtime implementation...")

if globals().get("STAGE56_RUNTIME_IMPL_ID") != (
    "v6-stage56-hardened-provenance-reuse-validate-20260919"
):
    raise RuntimeError(
        "STALE RUNTIME: exact v6 Stage5/6 implementation ID "
        "is not active. Run notebook definition cells from the beginning."
    )

if "build_feature_bank_outer" not in globals():
    raise RuntimeError(
        "STALE RUNTIME: build_feature_bank_outer is not active."
    )

if "validate_stage56_outer_complete" not in globals():
    raise RuntimeError(
        "STALE RUNTIME: validate_stage56_outer_complete is not active."
    )

if "repair_mislabeled_stage56_commits" not in globals():
    raise RuntimeError(
        "STALE RUNTIME: repair_mislabeled_stage56_commits is not active."
    )

if getattr(
    build_feature_bank_outer,
    "_stage56_runtime_impl_id",
    None
) != STAGE56_RUNTIME_IMPL_ID:
    raise RuntimeError(
        "STALE RUNTIME: active build_feature_bank_outer is not "
        "the exact hardened v6 implementation."
    )

if getattr(
    validate_stage56_outer_complete,
    "_stage56_runtime_impl_id",
    None
) != STAGE56_RUNTIME_IMPL_ID:
    raise RuntimeError(
        "STALE RUNTIME: active validate_stage56_outer_complete "
        "is not the exact hardened v6 implementation."
    )

active_names = set(build_feature_bank_outer.__code__.co_names)

if "repair_mislabeled_stage56_commits" not in active_names:
    raise RuntimeError(
        "STALE RUNTIME: active builder does not invoke "
        "Stage5/6 provenance repair."
    )

print("EXACT V6 STAGE5/6 RUNTIME IMPLEMENTATION: PASS")

RUN_LOG=[]
for outer in OUTER_FOLDS_TO_RUN:
    print("\n"+"="*120)
    print("START OUTER",outer,"| ResEncM -> graph -> GAT -> fixed Stage8/9 -> validated filter5 Stage11")
    print("="*120)

    print("Stage5/6 known-bug repair (pre-build)...")
    repair_mislabeled_stage56_commits(outer)

    build_feature_bank_outer(outer)

    print("Stage5/6 known-bug repair (post-build)...")
    repair_mislabeled_stage56_commits(outer)

    validate_stage56_outer_complete(outer)
    stage_outer_feature_bank_local(outer)

    print("GAT ...")
    gat_result=tune_refit_gat(outer)
    print("Fusion ...")
    fusion_result=tune_refit_fusion(outer)
    print("Outer prediction ...")
    pred_rows=predict_outer_fold(outer)

    RUN_LOG.append({
        "outer":outer,
        "gat_selected_epoch":gat_result["selected_epoch"],
        "fusion_selected_epoch":fusion_result["selected_epoch"],
        "n_outer_predictions":len(pred_rows),
    })

    print("OUTER",outer,"COMPLETE")

    _CASE_LRU.clear()
    _GAT_LRU.clear()
    _GT_LRU.clear()
    torch.cuda.empty_cache()
    gc.collect()

display(pd.DataFrame(RUN_LOG))
print("REQUESTED OUTER FOLDS COMPLETE")


In [ ]:
#@title 21. Stage16 metrics — raw vs validated Stage11 filter5 for CNN / GAT / Hybrid

def confusion(gt,pred):
    gt=np.asarray(gt,bool);pred=np.asarray(pred,bool)
    return (
        int(np.logical_and(gt,pred).sum()),
        int(np.logical_and(~gt,pred).sum()),
        int(np.logical_and(gt,~pred).sum()),
        int(np.logical_and(~gt,~pred).sum()),
    )

def safe_div(a,b,empty=1.):return float(a/b) if b else float(empty)

def hd95_mm(gt,pred,spacing_xyz):
    gt=np.asarray(gt,bool);pred=np.asarray(pred,bool);sp=np.asarray(tuple(spacing_xyz)[::-1],float)
    if not gt.any() and not pred.any():return 0.
    diag=float(np.sqrt(np.sum(((np.asarray(gt.shape)-1)*sp)**2)))
    if not gt.any() or not pred.any():return diag
    st=np.ones((3,3,3),bool)
    gs=np.logical_xor(gt,ndi.binary_erosion(gt,structure=st,border_value=0))
    ps=np.logical_xor(pred,ndi.binary_erosion(pred,structure=st,border_value=0))
    d1=ndi.distance_transform_edt(~ps,sampling=sp)[gs]
    d2=ndi.distance_transform_edt(~gs,sampling=sp)[ps]
    return float(np.percentile(np.concatenate([d1,d2]),95))

def metrics(gt,pred,spacing):
    tp,fp,fn,tn=confusion(gt,pred)
    return {
        "DSC":safe_div(2*tp,2*tp+fp+fn),
        "IoU":safe_div(tp,tp+fp+fn),
        "Sensitivity":safe_div(tp,tp+fn),
        "Specificity":safe_div(tn,tn+fp),
        "HD95_mm":hd95_mm(gt,pred,spacing),
    }

rows=[]
for outer in range(5):
    rr=predictions_remote(outer)
    for case in REG.loc[REG.fold==outer,"case"]:
        local=fold_local(outer)/f"{case}.outer_pred.npz"
        if not local.exists():
            x=(
                remote_file(rr,f"{case}.outer_pred.npz")
                or remote_file(rr,f"{case}.npz")
            )
            if not x:raise FileNotFoundError(("outer prediction",outer,case))
            download(x["id"],local)
        with np.load(local) as z:
            if (
                "protocol_sha" not in z.files
                or str(z["protocol_sha"].item())!=DOWNSTREAM_PROTOCOL_SHA
                or "implementation_id" not in z.files
                or str(z["implementation_id"].item())!=FUSION_IMPL_ID
            ):
                raise RuntimeError(("prediction protocol/implementation mismatch",outer,case))
            probs={k:z[k].astype(np.float32) for k in ["cnn","gat","hybrid"]}
            spacing=tuple(float(v) for v in z["spacing_xyz"])
        im=sitk.ReadImage(str(ensure_orig_gt(case)));gt=sitk.GetArrayFromImage(im)>0

        for branch,p in probs.items():
            for variant,pred in [
                ("Raw_p0.5",p>=0.5),
                ("Validated_Stage11_filter5",validated_stage11_filter5(p)),
            ]:
                rec={"case":case,"fold":outer,"branch":branch,"variant":variant}
                rec.update(metrics(gt,pred,spacing));rows.append(rec)

CASE_METRICS=pd.DataFrame(rows)
FOLD_METRICS=CASE_METRICS.groupby(["variant","branch","fold"],as_index=False)[
    ["DSC","IoU","Sensitivity","Specificity","HD95_mm"]
].mean()

summ=[]
for (variant,branch),g in FOLD_METRICS.groupby(["variant","branch"]):
    for metric in ["DSC","IoU","Sensitivity","Specificity","HD95_mm"]:
        v=g[metric].to_numpy(float)
        summ.append({
            "Variant":variant,"Branch":branch,"Metric":metric,
            "5Fold_Mean":float(v.mean()),"5Fold_Std":float(v.std(ddof=1)),
            "Mean_PlusMinus_Std":f"{v.mean():.6f} ± {v.std(ddof=1):.6f}",
        })
SUMMARY=pd.DataFrame(summ)

display(FOLD_METRICS)
display(SUMMARY)

for name,df in [
    ("V3_5_STAGE16_PER_CASE.csv",CASE_METRICS),
    ("V3_5_STAGE16_PER_FOLD.csv",FOLD_METRICS),
    ("V3_5_STAGE16_5FOLD_MEAN_STD.csv",SUMMARY),
]:
    p=REPORT_LOCAL/name;df.to_csv(p,index=False);upload(p,REMOTE_REPORT,name,"text/csv")


In [ ]:
#@title 22. Final provenance + result block

def sr(variant,branch,metric):
    q=SUMMARY[
        (SUMMARY.Variant==variant)&
        (SUMMARY.Branch==branch)&
        (SUMMARY.Metric==metric)
    ]
    assert len(q)==1
    return q.iloc[0]

manifest={
    "experiment":"GraphMS-Net ResEncM-250 TRUE HYBRID v3.5.1 FINAL GUIDE-LEAN AUDITED",
    "downstream_protocol_sha":DOWNSTREAM_PROTOCOL_SHA,
    "frozen_stage5_6_protocol_sha":LEGACY_FEATURE_PROTOCOL_SHA,
    "cnn":"ResEncM-250",
    "historical_m150_catmil_artifacts_used_for_model_inputs":False,
    "historical_handoff_usage":"case/patient/fold IDs only",
    "stage7":"GAT/message passing; FP32; baseline-preserving epoch selection; exact compatible checkpoint reuse allowed",
    "stage8":"CNN+GNN concat + SE + self-attention + multi-scale fusion",
    "stage9":"transposed-convolution decoder + CNN-logit skip + final 1x1x1 lesion head",
    "stage10":"0.5 Dice + 0.5 BCE",
    "stage11":{
        "threshold":GUIDE_THRESHOLD,
        "connected_components":"3D 26-connectivity",
        "remove_components_lt_voxels":GUIDE_MIN_COMPONENT_VOXELS,
        "setting_name":"filter5",
        "setting_source":"previously validated project setting; not claimed guide-mandated",
        "parameter_search":False,
        "morphology":"no unspecified morphology parameters invented",
    },
    "stage15":{
        "optimizer":"AdamW",
        "lr":GUIDE_LR,
        "betas":GUIDE_BETAS,
        "eps":GUIDE_EPS,
        "weight_decay":GUIDE_WEIGHT_DECAY,
        "schedule":"cosine",
        "cosine_horizon_epochs":FUSION_MAX_TUNE_EPOCHS,
        "min_lr":GUIDE_MIN_LR,
        "dropout":GUIDE_DROPOUT,
        "patch":"64^3",
        "batch":BATCH_SIZE,
        "gradient_accumulation":GRAD_ACCUM_STEPS,
        "iterations_per_epoch":FUSION_ITERS_PER_EPOCH,
        "partial_accumulation_normalization":"actual group size",
        "max_epochs":FUSION_MAX_TUNE_EPOCHS,
        "validation_every_epochs":FUSION_EVAL_EVERY,
        "early_stop_patience_epochs":FUSION_EARLY_STOP_PATIENCE_EPOCHS,
        "checkpoint_every_epoch":True,
        "amsgrad":False,
        "warmup":False,
        "stochastic_depth":False,
        "label_smoothing":False,
    },
    "search_workload":{
        "fusion_search":False,
        "loss_search":False,
        "optimizer_search":False,
        "scheduler_search":False,
        "stage11_grid_search":False,
    },
    "stage7_training":{"max_epochs":GAT_MAX_TUNE_EPOCHS,"validation_every_epochs":GAT_EVAL_EVERY,"early_stop_patience_epochs":GAT_EARLY_STOP_PATIENCE_EPOCHS},
    "selection_integrity":{
        "outer_gt_used_before_prediction":False,
        "stage7_tune":"train3 -> inner",
        "stage7_refit":"four non-outer folds",
        "stage8_tune":"train3 -> inner using tune GAT",
        "stage8_refit":"four non-outer folds using refit GAT",
    },
    "known_alignment_limits":{
        "active_dataset":"MSLesSeg",
        "active_modalities":["FLAIR","T1","T2"],
        "missing_modalities_fabricated":False,
        "preprocessing":"frozen ResEncM uses its validated nnU-Net plan-driven preprocessing",
        "stages2_3":"frozen ResEncM/nnU-Net preprocessing+augmentation lineage; literal BET/N4/ANTs chain is not falsely claimed",
        "stages12_14":"run after segmentation is frozen",
    },
    "metrics":SUMMARY.to_dict(orient="records"),
    "scope":"development 5-fold CV; no untouched external-test claim",
}

mp=REPORT_LOCAL/"FINAL_V3_5_1_MANIFEST.json"
write_json(mp,manifest)
upload(mp,REMOTE_REPORT,"FINAL_V3_5_1_MANIFEST.json","application/json")

print("\n"+"="*136)
print("GRAPHMS RESENCM-250 TRUE HYBRID v3.5.1 — FINAL GUIDE-LEAN AUDITED")
print("="*136)
print("Frozen Stage5/6 reused               : YES — SHA VERIFIED")
print("Stage7 exact-compatible reuse        : ALLOWED — PROVENANCE VERIFIED")
print("Single Stage8 path                   : concat + SE + self-attention + multi-scale")
print("Stage10                              : Dice+BCE")
print("Stage15                              : AdamW + cosine + dropout + weight decay")
print("Fusion iterations/epoch              :",FUSION_ITERS_PER_EPOCH)
print("Fusion max epochs                    :",FUSION_MAX_TUNE_EPOCHS)
print("Stage7 early-stop patience           :",GAT_EARLY_STOP_PATIENCE_EPOCHS)
print("Fusion early-stop patience           :",FUSION_EARLY_STOP_PATIENCE_EPOCHS)
print("Fusion checkpoint cadence            : EVERY COMPLETED EPOCH")
print("Partial accumulation normalization   : ACTUAL GROUP SIZE")
print("Fusion/loss/optimizer tournaments    : NO")
print("Stage11 parameter sweep              : NO")
print("Stage11                              : p=0.5 + 26-connected filter5")
print("Stage11 numeric setting guide-mandated: NO — PREVIOUSLY VALIDATED PROJECT SETTING")
print("Label smoothing / stochastic depth   : NO")
print("AMSGrad / warmup                     : NO")
print("Outer GT used before prediction      : NO")
print("-"*136)

for variant in ["Raw_p0.5","Validated_Stage11_filter5"]:
    print(variant)
    for branch in ["cnn","gat","hybrid"]:
        r=sr(variant,branch,"DSC")
        print(f"  {branch:<8} DSC : {r['5Fold_Mean']:.6f} ± {r['5Fold_Std']:.6f}")

print("-"*136)
print("FINAL HYBRID — VALIDATED STAGE11 FILTER5")
for metric in ["DSC","IoU","Sensitivity","Specificity","HD95_mm"]:
    r=sr("Validated_Stage11_filter5","hybrid",metric)
    print(f"  {metric:<12}: {r['5Fold_Mean']:.6f} ± {r['5Fold_Std']:.6f}")

print("-"*136)
print("Claim scope                          : DEVELOPMENT 5-FOLD CV")
print("External untouched-test claim        : NO")
print("Historical promotion gates           : NO")
print("Stages 12-14 silently claimed final  : NO")
print("NEXT                                  : FREEZE SEGMENTATION -> STAGE12 -> STAGE13 -> STAGE14 -> FINAL STAGE16")
print("="*136)
print("SEND ME THIS FINAL BLOCK + THE EXECUTED NOTEBOOK.")


In [ ]:
#@title 23. Package reports for download
PACKAGE=Path("/content/GraphMS_RESENCM250_TRUE_HYBRID_v3_5_1_RESULTS")
ZIP_PATH=Path(str(PACKAGE)+".zip")
ZIP_PATH.unlink(missing_ok=True)
shutil.make_archive(str(PACKAGE),"zip",root_dir=str(REPORT_LOCAL))
print("Local report ZIP:",ZIP_PATH)
print("Persistent namespace:",DOWNSTREAM_REMOTE_EXPERIMENT_NAME)
